In [1]:
import sys
print(sys.executable)

/Users/prteeja/Downloads/FT/llm-finetuning-lab/.venv/bin/python


In [2]:
import sys
import torch
import transformers
import datasets
import peft

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)

Python: /Users/prteeja/Downloads/FT/llm-finetuning-lab/.venv/bin/python
PyTorch: 2.13.0
Transformers: 5.16.1
Datasets: 5.0.1
PEFT: 0.20.0


# LLM Fine-Tuning Lab

## Project Goal

Fine-tune a small language model to convert natural-language
requests into structured JSON.

This notebook documents the concepts, mathematics, experiments,
implementation, and observations throughout the project.

# 1. What Is a Language Model?

## 1.1 Foundational Definition

A language model assigns probabilities to sequences of tokens.

For an autoregressive language model, the fundamental task is next-token
prediction:

P(x_t | x_1, x_2, ..., x_{t-1})

In words:

Given all previous tokens, predict the probability distribution
of the next token.

### Basic pipeline

Human language
    ↓
Tokens
    ↓
Token IDs
    ↓
Neural network
    ↓
Probability distribution over vocabulary
    ↓
Next token

### Training

The model starts with parameters (weights) that have not learned the
desired language patterns.

During training:

Input tokens
    ↓
Model prediction
    ↓
Compare prediction with target token
    ↓
Loss
    ↓
Backpropagation
    ↓
Gradients
    ↓
Update weights

This process is repeated over many training examples.

### Mathematical foundation

For an autoregressive model:

P(x_1, ..., x_n)
= ∏ P(x_t | x_1, ..., x_{t-1})

The model therefore learns to estimate:

P(x_t | x_{<t})

The fundamental pre-training objective is next-token prediction.

## 1.2 Token IDs and Embeddings

Token IDs are discrete identifiers, not meaningful numerical values.

For example:

cat → 10
dog → 11
car → 12

The model therefore maps each token ID to a learned continuous vector
called an embedding.

An embedding is a learned mapping:

E: V → R^d

where V is the vocabulary and d is the embedding dimension.

For a vocabulary of 50,000 tokens and an embedding dimension of 768,
the embedding matrix has shape:

50,000 × 768

The token ID is used to select a row from this matrix.

Example:

dog → token ID 11

Embedding lookup:

11 → [0.3, 0.5, 0.1, ...]

The embedding values are learned during training. We do not manually
assign semantic meanings to individual dimensions.

Pipeline so far:

Text
 ↓
Tokens
 ↓
Token IDs
 ↓
Embedding lookup
 ↓
Continuous vectors

## 1.3 Experiment: Embedding Lookup

We will create a tiny vocabulary and an embedding table using PyTorch.

The experiment demonstrates:

token ID → embedding lookup → vector

The embedding values are initially random. They have no meaningful
semantic interpretation yet because the model has not been trained.

In [3]:
import torch

# Our tiny vocabulary
vocab = {
    "cat": 0,
    "dog": 1,
    "car": 2
}

# Create an embedding layer:
# 3 tokens, each represented by 4 numbers
embedding = torch.nn.Embedding(
    num_embeddings=3,
    embedding_dim=4
)

print(embedding.weight)

Parameter containing:
tensor([[ 0.1482, -1.9941, -0.1885,  0.5337],
        [ 1.0233, -0.9585, -0.9117, -0.1742],
        [ 1.4446, -0.9978,  0.1750,  0.3080]], requires_grad=True)


In [4]:
dog_id = torch.tensor([vocab["dog"]])

dog_embedding = embedding(dog_id)

print("Token ID:", dog_id)
print("Embedding:", dog_embedding)
print("Shape:", dog_embedding.shape)

Token ID: tensor([1])
Embedding: tensor([[ 1.0233, -0.9585, -0.9117, -0.1742]], grad_fn=<EmbeddingBackward0>)
Shape: torch.Size([1, 4])


In [10]:
embedding.weight[1]

tensor([ 1.0233, -0.9585, -0.9117, -0.1742], grad_fn=<SelectBackward0>)

### Observation

`torch.nn.Embedding(3, 4)` created a 3 × 4 learnable parameter matrix.

Each row corresponds to one token, and each row contains the token's
4-dimensional representation.

The values are initially random because the embedding has not been
trained yet.

`requires_grad=True` means the embedding values are trainable parameters
that can be updated through gradient-based optimization.

An embedding lookup selects the row corresponding to the token ID.

`grad_fn` indicates that PyTorch is tracking the operation so gradients
can later be propagated backward through it.

## 1.4 How Does the Model Learn?

A neural network learns by repeatedly:

1. Making a prediction
2. Calculating a loss
3. Computing gradients through backpropagation
4. Updating its parameters using an optimizer

### Loss

A loss function measures how different the model's prediction is from
the desired target.

For language models, cross-entropy loss is commonly used for
next-token prediction.

Good prediction → low loss

Bad prediction  → high loss

### Gradient

A gradient tells us how the loss changes when a parameter changes.

For a parameter w:

∂L/∂w

This tells us the direction and magnitude of the local change in loss
with respect to that parameter.

### Gradient Descent

The basic update rule is:

w_new = w_old - η(∂L/∂w)

where η is the learning rate.

### Training loop

Input

 ↓

Model prediction

 ↓

Loss

 ↓

Backpropagation

 ↓

Gradients

 ↓

Optimizer

 ↓

Updated parameters

 ↓
 
Repeat

The parameters of an LLM, including its embedding parameters, are
updated through this process during training.

## 1.5 Experiment: Gradient Descent on One Parameter

We will train a model containing only one parameter.

The goal is to make the parameter learn the value 5.

This demonstrates the fundamental training loop:

parameter → prediction → loss → gradient → parameter update

In [18]:
import torch

# Start with the wrong value
w = torch.tensor(2.0, requires_grad=True)

# Our target
target = torch.tensor(5.0)

# Simple training loop
learning_rate = 0.1

for step in range(10):
    # Prediction
    prediction = w

    # Loss
    loss = (prediction - target) ** 2

    # Calculate gradient
    loss.backward()

    # Update parameter
    with torch.no_grad():
        w -= learning_rate * w.grad

    # Clear the old gradient
    w.grad.zero_()

    print(
        f"Step {step + 1}: "
        f"w = {w.item():.4f}, "
        f"loss = {loss.item():.4f}"
    )

Step 1: w = 2.6000, loss = 9.0000
Step 2: w = 3.0800, loss = 5.7600
Step 3: w = 3.4640, loss = 3.6864
Step 4: w = 3.7712, loss = 2.3593
Step 5: w = 4.0170, loss = 1.5099
Step 6: w = 4.2136, loss = 0.9664
Step 7: w = 4.3709, loss = 0.6185
Step 8: w = 4.4967, loss = 0.3958
Step 9: w = 4.5973, loss = 0.2533
Step 10: w = 4.6779, loss = 0.1621


In [19]:
w = torch.tensor(2.0, requires_grad=True)

loss = (w - 5) ** 2

loss.backward()

print("w:", w.item())
print("loss:", loss.item())
print("gradient:", w.grad.item())

w: 2.0
loss: 9.0
gradient: -6.0


### Observation

The parameter started at 2.0 and moved toward the target value 5.0
through repeated gradient-based updates.

The loss decreased as the parameter improved.

The gradient was -6.0 at w = 2.0, matching the analytical derivative:

dL/dw = 2(w - 5)

Backpropagation in PyTorch calculated this gradient automatically.

The optimizer/update rule then used the gradient to change the parameter.

This is the fundamental mechanism underlying neural network training.

## 1.6 Forward Pass and the Neural Network Training Cycle

A forward pass is the process of passing an input through the model to
produce a prediction.

Example:

x = 2
w = 1.5

y = wx
y = 3

If the target is 6:

L = (y - target)²
L = 9

The complete training cycle is:

Input

 ↓

Forward pass

 ↓

Prediction

 ↓

Loss

 ↓

Backward pass / backpropagation

 ↓

Gradients

 ↓

Optimizer

 ↓

Updated parameters

 ↓
 
Repeat

For a language model, the embedding produces the initial numerical
representation of the tokens. The Transformer then performs many
learned computations on these representations before producing
predictions.

Gradients can flow backward through those computations and update the
parameters, including the embedding parameters.

## 1.7 Logits, Softmax, and Probabilities

A language model's final neural-network output is a set of raw scores
called logits.

Logits are not probabilities.

Softmax converts logits into a probability distribution:

P_i = exp(z_i) / Σ exp(z_j)

The resulting probabilities are non-negative and sum to 1.

In [20]:
import torch

logits = torch.tensor([1.2, 0.3, 2.1, 3.4, -0.5])

probabilities = torch.softmax(logits, dim=0)

print("Logits:")
print(logits)

print("\nProbabilities:")
print(probabilities)

print("\nSum:")
print(probabilities.sum())

Logits:
tensor([ 1.2000,  0.3000,  2.1000,  3.4000, -0.5000])

Probabilities:
tensor([0.0765, 0.0311, 0.1881, 0.6903, 0.0140])

Sum:
tensor(1.)


In [21]:
tokens = ["cat", "dog", "runs", "sleeps", "car"]

predicted_index = torch.argmax(probabilities)

print("Predicted token:", tokens[predicted_index])
print("Probability:", probabilities[predicted_index].item())

Predicted token: sleeps
Probability: 0.6903092861175537


### Observation

The model does not directly output a word.

Its final output is a vector of logits, with one logit corresponding to
each token in the vocabulary.

Softmax converts these logits into a probability distribution.

The token with the highest probability is the most likely prediction,
although generation can use sampling rather than always selecting the
highest-probability token.

Pipeline:

Input → Transformer → Logits → Softmax → Probabilities → Next token

## 1.8 Cross-Entropy Loss

For a single prediction, cross-entropy loss can be expressed as:

L = -log(P_correct)

where P_correct is the probability assigned by the model to the
correct target token.

Higher probability assigned to the correct token → lower loss.

Lower probability assigned to the correct token → higher loss.

In [22]:
import torch

probabilities = torch.tensor([0.05, 0.10, 0.15, 0.60, 0.10])

correct_token_index = 3

correct_probability = probabilities[correct_token_index]

loss = -torch.log(correct_probability)

print("Correct token probability:", correct_probability.item())
print("Cross-entropy loss:", loss.item())

Correct token probability: 0.6000000238418579
Cross-entropy loss: 0.5108255743980408


### Observation

For a single example, cross-entropy loss is:

L = -log(P_correct)

The loss depends on the probability assigned to the correct target.

High probability for the correct token → low loss.
Low probability for the correct token → high loss.

During practical LLM training, the model produces logits and
CrossEntropyLoss operates on those logits directly.

The loss provides the training signal that is propagated backward
through the model to calculate gradients.

# 1.9 Transformer Architecture

A Transformer is a neural-network architecture designed to process
sequences using attention mechanisms.

Earlier recurrent architectures such as RNNs processed sequences
sequentially:

h_t = f(x_t, h_{t-1})

Transformers instead allow tokens to directly interact with other
tokens through self-attention.

For example, in:

"The cat sat on the mat because it was tired."

the representation of "it" can use information from other tokens in
the sequence to determine what "it" refers to.

Simplified decoder-only Transformer:

Input token IDs
 ↓
Token embeddings
 ↓
Positional information
 ↓
Transformer blocks
    ↓
    Self-attention
    ↓
    Feed-forward network
    ↓
    Normalization / residual connections
 ↓
Output projection
 ↓
Logits
 ↓
Probability distribution
 ↓
Next-token prediction

The core self-attention operation is:

Attention(Q, K, V)
= softmax(QKᵀ / √d_k)V

We will derive and understand each component rather than treating this
equation as a black box.

# 1.10 Self-Attention

Self-attention allows each token to dynamically determine which other
tokens in the sequence are relevant to its representation.

For each token, the model creates three learned representations:

Query (Q): what information the token is looking for.

Key (K): what type of information the token can be matched on.

Value (V): the information the token provides if selected.

For a given token, its Query is compared with the Keys of all tokens
using dot products:

QKᵀ

These produce relevance scores.

Softmax converts the scores into attention weights that are positive
and sum to 1.

The attention weights are then used to calculate a weighted combination
of the Value vectors.

The complete operation is:

Attention(Q,K,V)
= softmax(QKᵀ / √dₖ)V

Conceptually:

Query × Keys

    ↓

Relevance scores

    ↓

Softmax

    ↓

Attention weights

    ↓

Weighted Values

    ↓
    
New token representation

The Q, K and V transformations are produced using learned parameter
matrices W_Q, W_K and W_V.

### What Does the Final Attention Output Represent?

For the token **"love"**, the attention mechanism produced these weights:

- I → 0.10
- love → 0.20
- pizza → 0.20
- today → 0.49

We then use these weights to combine the **Value vectors**:

\[
0.10[1,2] + 0.20[2,3] + 0.20[4,1] + 0.49[1,5]
\]

which gives approximately:

\[
[1.79,\ 3.45]
\]

#### What do [1.79, 3.45] mean?

This is the **new contextual representation of the token "love" after attention**.

Before attention, "love" had its own representation. After attention, its representation now contains information gathered from the other tokens.

The two numbers `[1.79, 3.45]` are **not directly interpretable as individual concepts** such as:

- 1.79 = meaning of "love"
- 3.45 = meaning of "pizza"

Instead, they are coordinates in a learned representation space.

The important idea is:

\[
\boxed{\text{New representation of "love"} =
\text{weighted combination of information from relevant tokens}}
\]

Because "today" received a relatively high attention weight (0.49), its Value vector contributes strongly to the new representation.

In a real LLM, these vectors are much larger (for example, hundreds of dimensions), and the learned representations contain much richer contextual information.

So the complete flow is:

\[
\text{Token representation}
\rightarrow Q,K,V
\rightarrow \text{attention weights}
\rightarrow \text{weighted Values}
\rightarrow \boxed{\text{contextual representation}}
\]

This contextual representation is then passed to the next components of the Transformer block.

## 1.11 Experiment: Implementing Self-Attention

We will implement the self-attention calculation directly in PyTorch.

For simplicity, we will use the same 4-token example:

> I love pizza today

The goal is to calculate the contextual representation of **"love"** step by step.

The process is:

$$
Q_{love}K^T
\rightarrow
\frac{Q_{love}K^T}{\sqrt{d_k}}
\rightarrow
\text{softmax}
\rightarrow
\text{attention weights}
\rightarrow
\sum_i \text{weight}_i V_i
$$

This experiment uses manually specified Q, K, and V vectors so that we can clearly see the mathematics.

In a real Transformer, Q, K, and V are produced from the token representations using learned matrices:

$$
Q=XW_Q,\quad K=XW_K,\quad V=XW_V
$$

In [23]:
import torch

# Keys for our 4 tokens
K = torch.tensor([
    [1.0, 0.0],  # I
    [0.0, 1.0],  # love
    [1.0, 1.0],  # pizza
    [0.0, 2.0]   # today
])

# Values for our 4 tokens
V = torch.tensor([
    [1.0, 2.0],  # I
    [2.0, 3.0],  # love
    [4.0, 1.0],  # pizza
    [1.0, 5.0]   # today
])

# Query for "love"
Q_love = torch.tensor([0.0, 1.0])

print("Q_love:", Q_love)
print("K shape:", K.shape)
print("V shape:", V.shape)

Q_love: tensor([0., 1.])
K shape: torch.Size([4, 2])
V shape: torch.Size([4, 2])


In [24]:
# Compare the Query of "love" with every Key

scores = Q_love @ K.T

print("Raw attention scores:", scores)

Raw attention scores: tensor([0., 1., 1., 2.])


In [25]:
# Scale the attention scores

d_k = K.shape[1]

scaled_scores = scores / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

print("d_k:", d_k)
print("sqrt(d_k):", torch.sqrt(torch.tensor(d_k, dtype=torch.float32)))
print("Scaled attention scores:", scaled_scores)

d_k: 2
sqrt(d_k): tensor(1.4142)
Scaled attention scores: tensor([0.0000, 0.7071, 0.7071, 1.4142])


In [26]:
# Convert scaled scores into attention weights

attention_weights = torch.softmax(scaled_scores, dim=0)

print("Attention weights:", attention_weights)
print("Sum of weights:", attention_weights.sum())

Attention weights: tensor([0.1091, 0.2212, 0.2212, 0.4486])
Sum of weights: tensor(1.0000)


In [27]:
# Combine the Value vectors using the attention weights

context = attention_weights @ V

print("Contextual representation of 'love':", context)

Contextual representation of 'love': tensor([1.8847, 3.3457])


### Understanding the Final Attention Output

The attention weights tell us how strongly **"love"** attends to each token:

| Token | Attention Weight |
|---|---:|
| I | 0.1091 |
| love | 0.2212 |
| pizza | 0.2212 |
| today | 0.4486 |

These weights are then used to combine the corresponding Value vectors:

$$
0.1091[1,2]
+
0.2212[2,3]
+
0.2212[4,1]
+
0.4486[1,5]
$$

The result is:

$$
\boxed{[1.8847,\ 3.3457]}
$$

This is the **contextual representation of "love" after self-attention**.

It is not a probability or a prediction. It is a new vector representing "love" after incorporating information from the surrounding tokens.

The important distinction is:

- The **embedding** is the initial representation of the token.
- The **attention output** is a context-dependent representation.
- The attention output is **computed during the forward pass** using the model's learned parameters.
- The values `[1.8847, 3.3457]` are coordinates in a learned representation space; the individual numbers do not have simple human-readable meanings.

The complete process we implemented is:

$$
Q_{love}K^T
\rightarrow
\frac{Q_{love}K^T}{\sqrt{d_k}}
\rightarrow
\text{softmax}
\rightarrow
\text{attention weights}
\rightarrow
\text{weighted sum of Values}
\rightarrow
\boxed{\text{contextual representation}}
$$

In a real Transformer, this process happens for **every token in the sequence**, not just "love".

This is what we will implement next: calculating self-attention for all tokens simultaneously and examining the resulting **attention matrix**.

## 1.12 Self-Attention for All Tokens

In the previous experiment, we calculated attention only for the token **"love"**.

A Transformer does this for **every token simultaneously**.

Instead of having one Query:

$$
Q_{love}
$$

we use the Query matrix containing the Query for every token:

$$
Q =
\begin{bmatrix}
Q_I \\
Q_{love} \\
Q_{pizza} \\
Q_{today}
\end{bmatrix}
$$

We then calculate:

$$
QK^T
$$

This produces a matrix where:

- **Each row** represents the token doing the attending.
- **Each column** represents the token being attended to.

For our 4-token example, the result is a **4 × 4 attention score matrix**.

This is called the **attention matrix**.

The important idea is that self-attention allows every token to compare itself with every other token in the sequence.

In [28]:
# Queries for all 4 tokens
Q = torch.tensor([
    [1.0, 0.0],  # I
    [0.0, 1.0],  # love
    [1.0, 1.0],  # pizza
    [0.0, 2.0]   # today
])

# Calculate raw attention scores for all tokens
all_scores = Q @ K.T

print("Raw attention scores:")
print(all_scores)
print("Shape:", all_scores.shape)

Raw attention scores:
tensor([[1., 0., 1., 0.],
        [0., 1., 1., 2.],
        [1., 1., 2., 2.],
        [0., 2., 2., 4.]])
Shape: torch.Size([4, 4])


In [29]:
# Scale the scores for all tokens

d_k = K.shape[1]

scaled_all_scores = all_scores / torch.sqrt(
    torch.tensor(d_k, dtype=torch.float32)
)

# Apply softmax across each row
attention_matrix = torch.softmax(scaled_all_scores, dim=1)

print("Attention weights matrix:")
print(attention_matrix)

print("\nRow sums:")
print(attention_matrix.sum(dim=1))

Attention weights matrix:
tensor([[0.3349, 0.1651, 0.3349, 0.1651],
        [0.1091, 0.2212, 0.2212, 0.4486],
        [0.1651, 0.1651, 0.3349, 0.3349],
        [0.0382, 0.1573, 0.1573, 0.6471]])

Row sums:
tensor([1.0000, 1.0000, 1.0000, 1.0000])


In [30]:
# Combine the Value vectors using the attention weights

context_matrix = attention_matrix @ V

print("Contextual representations:")
print(context_matrix)
print("Shape:", context_matrix.shape)

Contextual representations:
tensor([[2.1698, 2.3256],
        [1.8847, 3.3457],
        [2.1698, 2.8349],
        [1.6293, 3.9413]])
Shape: torch.Size([4, 2])


## 1.12 Complete Self-Attention Calculation

We will now put together everything we learned and implemented in the self-attention experiment.

We use the same 4-token example:

> I love pizza today

### Starting Token Representations

| Token | Representation |
|---|---|
| I | `[1, 0]` |
| love | `[0, 1]` |
| pizza | `[1, 1]` |
| today | `[0, 2]` |

In a real Transformer, these token representations are transformed into Query, Key, and Value vectors using three different learned projection matrices:

$$
Q = XW_Q
$$

$$
K = XW_K
$$

$$
V = XW_V
$$

Here:

- $X$ = token representations
- $W_Q$ = learned Query projection matrix
- $W_K$ = learned Key projection matrix
- $W_V$ = learned Value projection matrix

Therefore:

$$
X \rightarrow Q,\ K,\ V
$$

For this simplified experiment, we manually specified Q, K, and V so that we could focus on understanding the attention mechanism.

---

### Step 1: Calculate Query-Key Scores

We calculate:

$$
QK^T
$$

This compares every Query with every Key.

Our result was:

| Query \ Key | I | love | pizza | today |
|---|---:|---:|---:|---:|
| I | 1 | 0 | 1 | 0 |
| love | 0 | 1 | 1 | 2 |
| pizza | 1 | 1 | 2 | 2 |
| today | 0 | 2 | 2 | 4 |

Each **row** represents the token that is looking (Query).

Each **column** represents the token being looked at (Key).

Therefore:

$$
\text{Rows} = \text{Queries}
$$

$$
\text{Columns} = \text{Keys}
$$

For example, the row for `"love"` is:

$$
[0,\ 1,\ 1,\ 2]
$$

meaning:

$$
\text{love} \rightarrow \text{I} = 0
$$

$$
\text{love} \rightarrow \text{love} = 1
$$

$$
\text{love} \rightarrow \text{pizza} = 1
$$

$$
\text{love} \rightarrow \text{today} = 2
$$

These are called the **raw attention scores**.

---

### Step 2: Scale the Scores

We divide the scores by:

$$
\sqrt{d_k}
$$

where $d_k$ is the dimension of each Key vector.

In our example:

$$
d_k = 2
$$

so:

$$
\sqrt{d_k} = \sqrt{2} \approx 1.4142
$$

For the `"love"` row:

$$
[0,\ 1,\ 1,\ 2] / \sqrt{2}
$$

which gives:

$$
[0,\ 0.7071,\ 0.7071,\ 1.4142]
$$

The scaling prevents dot products from becoming excessively large as the dimensionality increases.

---

### Step 3: Apply Softmax

We apply softmax to each row:

$$
\text{Attention weights}
=
\text{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)
$$

This converts the scores into weights that sum to 1 for each token.

For `"love"`, we obtained:

| Token | Attention Weight |
|---|---:|
| I | 0.1091 |
| love | 0.2212 |
| pizza | 0.2212 |
| today | 0.4486 |

These weights determine how strongly `"love"` attends to each token.

For example, `"today"` receives the highest attention weight:

$$
0.4486
$$

so its Value vector contributes more strongly to the final contextual representation of `"love"`.

---

### Step 4: Combine the Value Vectors

The attention weights are used to calculate a weighted sum of the Value vectors:

$$
\text{Context}
=
\text{Attention weights} \times V
$$

For `"love"`:

$$
0.1091[1,2]
+
0.2212[2,3]
+
0.2212[4,1]
+
0.4486[1,5]
$$

which produces:

$$
\boxed{[1.8847,\ 3.3457]}
$$

This is the **contextual representation of `"love"` after self-attention**.

It is not a probability or a prediction. It is a new vector representing `"love"` after incorporating information from the surrounding tokens.

---

### Step 5: Contextual Representations for All Tokens

A Transformer performs the same operation for every token simultaneously.

Our final contextual representations were:

| Token | Before Attention | After Attention |
|---|---|---|
| I | `[1, 0]` | `[2.1698, 2.3256]` |
| love | `[0, 1]` | `[1.8847, 3.3457]` |
| pizza | `[1, 1]` | `[2.1698, 2.8349]` |
| today | `[0, 2]` | `[1.6293, 3.9413]` |

The representations changed because each token gathered information from the other tokens according to its attention weights.

The `"love"` row matches the result we calculated earlier:

$$
[1.8847,\ 3.3457]
$$

This confirms that calculating attention for `"love"` individually and calculating attention for all tokens simultaneously produce the same result.

---

### Complete Self-Attention Equation

All the steps can be written as one equation:

$$
\boxed{
\text{Attention}(Q,K,V)
=
\text{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
}
$$

The complete flow is:

$$
QK^T
\rightarrow
\text{scaling}
\rightarrow
\text{softmax}
\rightarrow
\text{attention weights}
\rightarrow
\text{weighted Values}
\rightarrow
\text{contextual representations}
$$

### Core Intuition

**Query and Key determine where to look:**

$$
QK^T
\rightarrow
\text{relevance scores}
$$

**Softmax determines how much to look:**

$$
\text{softmax}
\rightarrow
\text{attention weights}
$$

**Value determines what information to take:**

$$
\text{attention weights} \times V
\rightarrow
\text{contextual representation}
$$

Therefore:

$$
\boxed{
Q + K \rightarrow \text{Where to look}
}
$$

$$
\boxed{
V \rightarrow \text{What information to take}
}
$$

The key idea is:

> Each token starts with its own representation and uses self-attention to selectively incorporate information from other tokens, producing a new context-dependent representation.

We have now implemented the mathematical core of self-attention in PyTorch.

The next question is:

> How does an autoregressive LLM prevent a token from looking at future tokens?

This is solved using **causal masking**.

## 1.13 Causal Masking

Self-attention, by itself, allows every token to attend to every other token.

For example, with:

> I love pizza today

ordinary self-attention allows:

$$
\text{I} \leftrightarrow \text{love} \leftrightarrow \text{pizza} \leftrightarrow \text{today}
$$

However, an autoregressive language model has a specific job:

> **Predict the next token using only the tokens that have already appeared.**

Therefore, when predicting a token, the model must not be allowed to look at future tokens.

### Why Is This Necessary?

Suppose the training sentence is:

> I love pizza

The model creates training examples like:

| Context | Target |
|---|---|
| `I` | `love` |
| `I love` | `pizza` |

When predicting `love`, the model should only have access to:

$$
[I]
$$

When predicting `pizza`, the model should only have access to:

$$
[I,\ love]
$$

It would be cheating if the model could see `pizza` while trying to predict `pizza`.

Therefore, during training, the model must enforce:

$$
\boxed{
\text{A token can attend to itself and tokens before it, but not future tokens.}
}
$$

### Visual Representation

For the sequence:

> I love pizza today

the allowed attention pattern is:

| Query \ Key | I | love | pizza | today |
|---|---:|---:|---:|---:|
| I | ✓ | ✗ | ✗ | ✗ |
| love | ✓ | ✓ | ✗ | ✗ |
| pizza | ✓ | ✓ | ✓ | ✗ |
| today | ✓ | ✓ | ✓ | ✓ |

Think of each row as asking:

> **"Which tokens am I allowed to look at?"**

The first token `"I"` can only look at itself.

The second token `"love"` can look at `"I"` and `"love"`.

The third token `"pizza"` can look at `"I"`, `"love"`, and `"pizza"`.

The fourth token `"today"` can look at all four tokens.

This produces a **lower-triangular attention pattern**:

$$
\begin{bmatrix}
1 & 0 & 0 & 0 \\
1 & 1 & 0 & 0 \\
1 & 1 & 1 & 0 \\
1 & 1 & 1 & 1
\end{bmatrix}
$$

Here:

- `1` = attention is allowed
- `0` = attention is blocked

### How Is It Actually Implemented?

The model does not normally remove the future tokens.

Instead, it modifies their attention scores **before softmax**.

The future positions are replaced with a very large negative number:

$$
-\infty
$$

For example:

$$
[0,\ 1,\ 1,\ 2]
$$

might become:

$$
[0,\ -\infty,\ -\infty,\ -\infty]
$$

before softmax.

Why does this work?

Because:

$$
e^{-\infty}=0
$$

Therefore, after softmax, the blocked positions receive an attention weight of exactly zero.

So the process is:

$$
\text{Raw scores}
\rightarrow
\text{Apply causal mask}
\rightarrow
\text{Softmax}
\rightarrow
\text{Future tokens receive weight }0
$$

### Important Distinction

The causal mask is applied to the **attention scores**, not to the token embeddings.

The tokens still exist and are still processed.

The mask simply prevents information from future positions from influencing the current token's attention output.

### Why Is This Called "Causal"?

It enforces the direction of information flow:

$$
\text{Past} \rightarrow \text{Present} \rightarrow \text{Future}
$$

Information can flow from earlier tokens to later tokens, but not backwards from future tokens to earlier tokens.

This matches the way autoregressive generation works.

At inference time, if the model has generated:

> `I love`

and needs to predict the next token, it has no future token available anyway.

The causal mask ensures that **training follows the same information constraint as generation**.

### Key Takeaway

Without causal masking:

$$
\text{Token can see past + present + future}
$$

With causal masking:

$$
\boxed{
\text{Token can see past + present only}
}
$$

This is what allows a decoder-only Transformer such as GPT-style models to learn next-token prediction without seeing the answer in advance.

In [31]:
# Create a causal mask
# True = allowed to attend
# False = not allowed to attend

causal_mask = torch.tril(
    torch.ones(4, 4, dtype=torch.bool)
)

print("Causal mask:")
print(causal_mask)

Causal mask:
tensor([[ True, False, False, False],
        [ True,  True, False, False],
        [ True,  True,  True, False],
        [ True,  True,  True,  True]])


In [34]:
# Apply the causal mask

masked_scores = scaled_all_scores.masked_fill(
    ~causal_mask,
    float("-inf")
)

print("Scaled scores before masking:")
print(scaled_all_scores)

print("\nScores after causal masking:")
print(masked_scores)

Scaled scores before masking:
tensor([[0.7071, 0.0000, 0.7071, 0.0000],
        [0.0000, 0.7071, 0.7071, 1.4142],
        [0.7071, 0.7071, 1.4142, 1.4142],
        [0.0000, 1.4142, 1.4142, 2.8284]])

Scores after causal masking:
tensor([[0.7071,   -inf,   -inf,   -inf],
        [0.0000, 0.7071,   -inf,   -inf],
        [0.7071, 0.7071, 1.4142,   -inf],
        [0.0000, 1.4142, 1.4142, 2.8284]])


In [35]:
# Convert masked scores into attention weights

masked_attention_weights = torch.softmax(
    masked_scores,
    dim=1
)

print("Causal attention weights:")
print(masked_attention_weights)

print("\nRow sums:")
print(masked_attention_weights.sum(dim=1))

Causal attention weights:
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.3302, 0.6698, 0.0000, 0.0000],
        [0.2483, 0.2483, 0.5035, 0.0000],
        [0.0382, 0.1573, 0.1573, 0.6471]])

Row sums:
tensor([1., 1., 1., 1.])


Attention weights matrix:
tensor([[0.3349, 0.1651, 0.3349, 0.1651],
        [0.1091, 0.2212, 0.2212, 0.4486],
        [0.1651, 0.1651, 0.3349, 0.3349],
        [0.0382, 0.1573, 0.1573, 0.6471]])

In [36]:
# Calculate contextual representations using causal attention

causal_context_matrix = masked_attention_weights @ V

print("Causal contextual representations:")
print(causal_context_matrix)
print("Shape:", causal_context_matrix.shape)

Causal contextual representations:
tensor([[1.0000, 2.0000],
        [1.6698, 2.6698],
        [2.7587, 1.7448],
        [1.6293, 3.9413]])
Shape: torch.Size([4, 2])


### 1.13.1 Complete Causal Masking Experiment

We previously calculated self-attention without restricting which tokens could see each other.

For the sequence:

> I love pizza today

ordinary self-attention allows every token to attend to every other token.

For an autoregressive language model, this would cause a problem: a token could see information from the future.

### The Causal Constraint

An autoregressive language model predicts the next token using the tokens that have already appeared.

Therefore, position $t$ can attend only to positions $\leq t$.

$$
\boxed{
\text{Position }t
\rightarrow
\text{positions }0,\ldots,t
}
$$

It cannot attend to positions after $t$.

For our sequence:

| Query \ Key | I | love | pizza | today |
|---|---:|---:|---:|---:|
| I | ✓ | ✗ | ✗ | ✗ |
| love | ✓ | ✓ | ✗ | ✗ |
| pizza | ✓ | ✓ | ✓ | ✗ |
| today | ✓ | ✓ | ✓ | ✓ |

This creates a lower-triangular pattern:

$$
\begin{bmatrix}
1 & 0 & 0 & 0 \\
1 & 1 & 0 & 0 \\
1 & 1 & 1 & 0 \\
1 & 1 & 1 & 1
\end{bmatrix}
$$

Here:

- `1` = attention is allowed
- `0` = attention is blocked

### Applying the Mask

We first calculate the normal attention scores:

$$
QK^T
$$

and scale them:

$$
\frac{QK^T}{\sqrt{d_k}}
$$

Before softmax, the scores corresponding to future positions are replaced with:

$$
-\infty
$$

For example, the `"love"` row becomes:

$$
[0,\ 0.7071,\ -\infty,\ -\infty]
$$

The mask is applied **before softmax**.

This works because:

$$
e^{-\infty}=0
$$

Therefore, after softmax, the masked positions receive an attention weight of zero.

### Causal Attention Weights

Our resulting attention weights were:

| Query \ Key | I | love | pizza | today |
|---|---:|---:|---:|---:|
| I | 1.0000 | 0 | 0 | 0 |
| love | 0.3302 | 0.6698 | 0 | 0 |
| pizza | 0.2483 | 0.2483 | 0.5035 | 0 |
| today | 0.0382 | 0.1573 | 0.1573 | 0.6471 |

Notice that every row still sums to 1.

The difference is that **future positions now have exactly zero attention**.

### Effect on Contextual Representations

We then calculated:

$$
\text{Context}
=
\text{Causal attention weights} \times V
$$

This produced:

| Token | Without Causal Mask | With Causal Mask |
|---|---|---|
| I | `[2.1698, 2.3256]` | `[1.0000, 2.0000]` |
| love | `[1.8847, 3.3457]` | `[1.6698, 2.6698]` |
| pizza | `[2.1698, 2.8349]` | `[2.7587, 1.7448]` |
| today | `[1.6293, 3.9413]` | `[1.6293, 3.9413]` |

The representations change because each token now has access only to its allowed context.

The `"today"` representation does not change because `"today"` is the final token and therefore has no future tokens to block.

### Why Can a Token Attend to Itself?

The causal rule is:

$$
\boxed{
\text{Position }t\text{ can attend to positions }\leq t
}
$$

The current token is therefore allowed.

This is because the representation at position $t$ is used to predict the **next token**, position $t+1$.

For example:

| Available Context | Target |
|---|---|
| `I` | `love` |
| `I love` | `pizza` |
| `I love pizza` | `today` |

So when predicting `pizza`, the model can use the representation at the `"love"` position, which has access to:

$$
[I,\ love]
$$

but not `"pizza"` itself or any later token.

### Information Flow

Without causal masking:

$$
\text{Past} + \text{Present} + \text{Future}
\rightarrow
\text{representation}
$$

With causal masking:

$$
\boxed{
\text{Past} + \text{Present}
\rightarrow
\text{representation}
}
$$

This prevents future information from leaking into earlier positions.

### Complete Autoregressive Attention Flow

The complete process is:

$$
X
\rightarrow
Q,K,V
\rightarrow
QK^T
\rightarrow
\frac{QK^T}{\sqrt{d_k}}
\rightarrow
\text{causal mask}
\rightarrow
\text{softmax}
\rightarrow
\text{attention weights}
\rightarrow
V
\rightarrow
\text{contextual representations}
$$

The final attention equation for a causal decoder-only Transformer can therefore be written conceptually as:

$$
\boxed{
\text{Causal Attention}(Q,K,V)
=
\text{softmax}
\left(
\frac{QK^T+\text{Mask}}{\sqrt{d_k}}
\right)V
}
$$

where the mask contains $-\infty$ at positions that are not allowed to be attended to.

### Key Takeaway

Self-attention answers:

> **Which tokens are relevant to me, and what information should I take from them?**

Causal masking adds an important constraint:

> **You may look at yourself and the past, but you may not look into the future.**

This is what makes self-attention suitable for autoregressive next-token prediction.

### Interpretation of Attention Output

The individual numbers in an attention output vector do not have a simple human-interpretable meaning.

For example:

$$
[1.000,\ 2.000]
$$

simply represents the new position/vector of the token in the model's representation space.

The important idea is:

$$
\boxed{
\text{Initial representation}
\rightarrow
\text{Self-Attention}
\rightarrow
\text{New contextual representation}
}
$$

The new representation contains information influenced by the tokens that were allowed to be attended to.

## 1.14.1 Experiment: Two Attention Heads

We will build a small multi-head attention example.

Our input contains:

- 4 tokens
- 4 dimensions per token
- 2 attention heads
- 2 dimensions per head

The important distinction is:

$$
\text{Sequence length} = 4
$$

$$
d_{model} = 4
$$

$$
d_{head} = 2
$$

The 4 tokens are **not divided between the heads**.

Both heads process all 4 tokens.

Instead, the 4-dimensional representation of each token is projected into a 2-dimensional representation inside each head.

Therefore:

$$
X:4\times4
$$

and for each head:

$$
Q_h,K_h,V_h:4\times2
$$

The attention score matrix is still:

$$
Q_hK_h^T
=
(4\times2)(2\times4)
=
4\times4
$$

because there are 4 tokens.

Each head therefore produces:

$$
\text{head}_h:4\times2
$$

The outputs from both heads are then concatenated:

$$
(4\times2)+(4\times2)
\rightarrow
4\times4
$$

Finally, the concatenated representation is passed through the output projection $W_O$.

In [37]:
import torch

# 4 tokens × 4 dimensions
X = torch.tensor([
    [0.2, 0.5, 0.1, 0.8],  # I
    [0.7, 0.1, 0.9, 0.3],  # love
    [0.4, 0.8, 0.2, 0.6],  # pizza
    [0.9, 0.3, 0.5, 0.2]   # today
])

d_model = 4
num_heads = 2
d_head = d_model // num_heads

print("X:")
print(X)

print("\nX shape:", X.shape)
print("d_model:", d_model)
print("num_heads:", num_heads)
print("d_head:", d_head)

X:
tensor([[0.2000, 0.5000, 0.1000, 0.8000],
        [0.7000, 0.1000, 0.9000, 0.3000],
        [0.4000, 0.8000, 0.2000, 0.6000],
        [0.9000, 0.3000, 0.5000, 0.2000]])

X shape: torch.Size([4, 4])
d_model: 4
num_heads: 2
d_head: 2


### 1.14.2 Per-Head Q, K, and V Projections

Both attention heads receive the complete input matrix $X$.

Each head has its own projection matrices:

$$
Q_h = XW_{Q,h}
$$

$$
K_h = XW_{K,h}
$$

$$
V_h = XW_{V,h}
$$

For our example:

$$
X: 4 \times 4
$$

$$
W_Q, W_K, W_V: 4 \times 2
$$

Therefore:

$$
Q,K,V: 4 \times 2
$$

The two heads see the same four tokens, but their different projection matrices allow them to create different representations and therefore learn different attention patterns.

In [38]:
# Projection matrices for Head 1
W_Q1 = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
    [0.5, 0.5]
])

W_K1 = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
    [0.5, 0.5]
])

W_V1 = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
    [0.5, 0.5]
])


# Projection matrices for Head 2
W_Q2 = torch.tensor([
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, -1.0],
    [0.5, -0.5]
])

W_K2 = torch.tensor([
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, -1.0],
    [0.5, -0.5]
])

W_V2 = torch.tensor([
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, -1.0],
    [0.5, -0.5]
])


# Create Q, K, V for each head
Q1 = X @ W_Q1
K1 = X @ W_K1
V1 = X @ W_V1

Q2 = X @ W_Q2
K2 = X @ W_K2
V2 = X @ W_V2


print("Head 1")
print("Q1:\n", Q1)
print("K1:\n", K1)
print("V1:\n", V1)

print("\nHead 2")
print("Q2:\n", Q2)
print("K2:\n", K2)
print("V2:\n", V2)

print("\nShapes:")
print("Q1:", Q1.shape)
print("K1:", K1.shape)
print("V1:", V1.shape)

print("Q2:", Q2.shape)
print("K2:", K2.shape)
print("V2:", V2.shape)

Head 1
Q1:
 tensor([[0.7000, 1.0000],
        [1.7500, 1.1500],
        [0.9000, 1.3000],
        [1.5000, 0.9000]])
K1:
 tensor([[0.7000, 1.0000],
        [1.7500, 1.1500],
        [0.9000, 1.3000],
        [1.5000, 0.9000]])
V1:
 tensor([[0.7000, 1.0000],
        [1.7500, 1.1500],
        [0.9000, 1.3000],
        [1.5000, 0.9000]])

Head 2
Q2:
 tensor([[ 1.0000, -0.3000],
        [ 1.1500, -0.3500],
        [ 1.3000, -0.1000],
        [ 0.9000,  0.3000]])
K2:
 tensor([[ 1.0000, -0.3000],
        [ 1.1500, -0.3500],
        [ 1.3000, -0.1000],
        [ 0.9000,  0.3000]])
V2:
 tensor([[ 1.0000, -0.3000],
        [ 1.1500, -0.3500],
        [ 1.3000, -0.1000],
        [ 0.9000,  0.3000]])

Shapes:
Q1: torch.Size([4, 2])
K1: torch.Size([4, 2])
V1: torch.Size([4, 2])
Q2: torch.Size([4, 2])
K2: torch.Size([4, 2])
V2: torch.Size([4, 2])


### 1.14.3 Attention Inside Head 1

Head 1 performs the same self-attention calculation we implemented earlier, but using its own projected Q, K, and V.

First calculate the attention scores:

$$
S_1 = Q_1K_1^T
$$

The shape is:

$$
(4\times2)(2\times4)=4\times4
$$

Then scale the scores:

$$
S_1'=\frac{S_1}{\sqrt{d_k}}
$$

where $d_k=2$.

Then apply softmax row-wise:

$$
A_1=\text{softmax}(S_1')
$$

Finally, calculate the contextual representations:

$$
H_1=A_1V_1
$$

The result $H_1$ has shape $4\times2$.

Each row represents the contextual representation of one token from the perspective of Head 1.

In [66]:
h1 = torch.softmax(((Q1 @ K1.T)/torch.sqrt(torch.tensor(d_head))), dim=1) @ V1
h2 = torch.softmax(((Q2 @ K2.T)/torch.sqrt(torch.tensor(d_head))), dim=1) @ V2
h = torch.concat((h1,h2), dim=1)


In [64]:
# Head 1 attention

d_k = Q1.shape[1]

scores1 = Q1 @ K1.T
scaled_scores1 = scores1 / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

attention_weights1 = torch.softmax(scaled_scores1, dim=1)

head1_output = attention_weights1 @ V1

print("Head 1 raw scores:")
print(scores1)

print("\nHead 1 scaled scores:")
print(scaled_scores1)

print("\nHead 1 attention weights:")
print(attention_weights1)

print("\nRow sums:")
print(attention_weights1.sum(dim=1))

print("\nHead 1 output:")
print(head1_output)

print("\nHead 1 output shape:")
print(head1_output.shape)

Head 1 raw scores:
tensor([[1.4900, 2.3750, 1.9300, 1.9500],
        [2.3750, 4.3850, 3.0700, 3.6600],
        [1.9300, 3.0700, 2.5000, 2.5200],
        [1.9500, 3.6600, 2.5200, 3.0600]])

Head 1 scaled scores:
tensor([[1.0536, 1.6794, 1.3647, 1.3789],
        [1.6794, 3.1007, 2.1708, 2.5880],
        [1.3647, 2.1708, 1.7678, 1.7819],
        [1.3789, 2.5880, 1.7819, 2.1637]])

Head 1 attention weights:
tensor([[0.1780, 0.3327, 0.2429, 0.2464],
        [0.1080, 0.4474, 0.1766, 0.2680],
        [0.1599, 0.3581, 0.2393, 0.2427],
        [0.1244, 0.4168, 0.1861, 0.2727]])

Row sums:
tensor([1.0000, 1.0000, 1.0000, 1.0000])

Head 1 output:
tensor([[1.2951, 1.0981],
        [1.4195, 1.0933],
        [1.3180, 1.1012],
        [1.3930, 1.0911]])

Head 1 output shape:
torch.Size([4, 2])


In [67]:
print("h1 shape:", h1.shape)
print("h2 shape:", h2.shape)
print("Combined shape:", h.shape)

print("\nHead 1 output:")
print(h1)

print("\nHead 2 output:")
print(h2)

print("\nCombined output:")
print(h)

h1 shape: torch.Size([4, 2])
h2 shape: torch.Size([4, 2])
Combined shape: torch.Size([4, 4])

Head 1 output:
tensor([[1.2951, 1.0981],
        [1.4195, 1.0933],
        [1.3180, 1.1012],
        [1.3930, 1.0911]])

Head 2 output:
tensor([[ 1.1070, -0.1370],
        [ 1.1099, -0.1405],
        [ 1.1098, -0.1320],
        [ 1.0986, -0.1104]])

Combined output:
tensor([[ 1.2951,  1.0981,  1.1070, -0.1370],
        [ 1.4195,  1.0933,  1.1099, -0.1405],
        [ 1.3180,  1.1012,  1.1098, -0.1320],
        [ 1.3930,  1.0911,  1.0986, -0.1104]])


### 1.14.4 Combining the Attention Heads

After each head performs self-attention, we have:

$$
H_1:4\times2
$$

$$
H_2:4\times2
$$

We concatenate the outputs along the feature dimension:

$$
H=\text{Concat}(H_1,H_2)
$$

Therefore:

$$
H:4\times4
$$

Concatenation puts the information from the heads side-by-side. It does not itself mix information between heads.

The Transformer then applies an output projection:

$$
Y=HW_O
$$

where:

$$
W_O:4\times4
$$

and therefore:

$$
Y:(4\times4)(4\times4)=4\times4
$$

The output projection is a learned linear transformation that mixes information from the different attention heads.

Therefore:

**Concatenation restores the total model dimension.**

**$W_O$ mixes and transforms the information produced by the heads.**

In [68]:
W_O = torch.tensor([
    [1.0, 0.2, 0.0, 0.1],
    [0.1, 1.0, 0.2, 0.0],
    [0.3, 0.0, 1.0, 0.2],
    [0.0, 0.3, 0.1, 1.0]
])

output = h @ W_O

print("Combined heads:")
print(h)

print("\nW_O:")
print(W_O)

print("\nFinal output:")
print(output)

print("\nFinal output shape:")
print(output.shape)

Combined heads:
tensor([[ 1.2951,  1.0981,  1.1070, -0.1370],
        [ 1.4195,  1.0933,  1.1099, -0.1405],
        [ 1.3180,  1.1012,  1.1098, -0.1320],
        [ 1.3930,  1.0911,  1.0986, -0.1104]])

W_O:
tensor([[1.0000, 0.2000, 0.0000, 0.1000],
        [0.1000, 1.0000, 0.2000, 0.0000],
        [0.3000, 0.0000, 1.0000, 0.2000],
        [0.0000, 0.3000, 0.1000, 1.0000]])

Final output:
tensor([[1.7370, 1.3161, 1.3130, 0.2139],
        [1.8618, 1.3350, 1.3145, 0.2235],
        [1.7611, 1.3252, 1.3169, 0.2218],
        [1.8317, 1.3366, 1.3058, 0.2486]])

Final output shape:
torch.Size([4, 4])


## 1.15 Residual Connections and Layer Normalization

### Residual Connections

A Transformer does not simply replace a token's representation with the output of a transformation.

Instead, it preserves the original representation and adds the newly computed information.

If a transformation is represented by $F(X)$:

$$
Y = X + F(X)
$$

For example:

$$
X=[2,4,6,8]
$$

and:

$$
F(X)=[1,0,2,-1]
$$

then:

$$
Y=[2,4,6,8]+[1,0,2,-1]
$$

$$
Y=[3,4,8,7]
$$

The original representation therefore has a direct path around the transformation.

This is called a **residual connection** or **skip connection**.

Residual connections are important because they:
- preserve information from earlier layers,
- allow a transformation to add or modify information rather than having to recreate everything,
- provide a relatively direct path for gradients during training.

---

### Normalization

Normalization generally means transforming numerical values into a more controlled numerical distribution or scale.

A common standardization is:

$$
z=\frac{x-\mu}{\sigma}
$$

where:
- $\mu$ is the mean,
- $\sigma$ is the standard deviation.

Normalization helps neural networks keep intermediate representations numerically well-behaved as information passes through many transformations.

---

### Layer Normalization

Layer Normalization (LayerNorm) applies normalization to the feature dimensions of an individual token representation.

For example, if a token has:

$$
X=[2,4,6,8]
$$

LayerNorm calculates statistics from these four feature values and normalizes them.

For a sequence:

$$
X:
\begin{bmatrix}
\text{token 1}\\
\text{token 2}\\
\text{token 3}\\
\text{token 4}
\end{bmatrix}
$$

LayerNorm operates independently on each token's representation.

Therefore, if the representation has shape:

$$
4\times4
$$

there are 4 token representations, and each token's 4 feature values are normalized independently.

LayerNorm also has learned parameters $\gamma$ and $\beta$:

$$
\text{LayerNorm}(x)
=
\gamma
\frac{x-\mu}{\sqrt{\sigma^2+\epsilon}}
+\beta
$$

where $\epsilon$ provides numerical stability.

---

### Residual Connection + LayerNorm

For an attention sublayer, conceptually:

$$
A=\text{Attention}(X)
$$

Then the residual connection combines the original representation with the attention output:

$$
R=X+A
$$

Then LayerNorm is applied:

$$
Y=\text{LayerNorm}(R)
$$

Therefore:

**Attention:** gathers contextual information from other tokens.

**Residual connection:** preserves the original representation and adds the newly computed information.

**LayerNorm:** normalizes each token's feature representation.

The same general pattern is also used around the feed-forward network (FFN), which will be studied separately.

### 1.15.1 Experiment: Layer Normalization

Consider one token representation:

$$
x=[2,4,6,8]
$$

LayerNorm normalizes across the feature dimensions of this token.

First calculate the mean:

$$
\mu=\frac{1}{d}\sum_{i=1}^{d}x_i
$$

Then calculate the variance:

$$
\sigma^2=\frac{1}{d}\sum_{i=1}^{d}(x_i-\mu)^2
$$

Then normalize each feature:

$$
\hat{x}_i=
\frac{x_i-\mu}
{\sqrt{\sigma^2+\epsilon}}
$$

Finally, LayerNorm applies learned scale and shift parameters:

$$
y_i=\gamma\hat{x}_i+\beta
$$

In this experiment, we will initially use $\gamma=1$ and $\beta=0$ so that we can see the normalization itself.

In [72]:
import torch

x = torch.tensor([2.0, 4.0, 6.0, 8.0])

mean = x.mean()
variance = ((x - mean) ** 2).mean()
epsilon = 1e-5

normalized = (x - mean)/torch.sqrt(variance + epsilon)

print("Original:", x)
print("Mean:", mean)
print("Variance:", variance)
print("Normalized:", normalized)
print("Normalized mean:", normalized.mean())
print("Normalized variance:", normalized.var(unbiased=False))

Original: tensor([2., 4., 6., 8.])
Mean: tensor(5.)
Variance: tensor(5.)
Normalized: tensor([-1.3416, -0.4472,  0.4472,  1.3416])
Normalized mean: tensor(0.)
Normalized variance: tensor(1.0000)


### 1.15.2 Experiment: PyTorch LayerNorm

PyTorch provides LayerNorm through `torch.nn.LayerNorm`.

For a token representation with 4 features:

$$
x=[2,4,6,8]
$$

we can create:

`torch.nn.LayerNorm(4)`

The `4` means that LayerNorm operates across the 4 feature dimensions.

LayerNorm first normalizes the representation:

$$
\hat{x}=
\frac{x-\mu}{\sqrt{\sigma^2+\epsilon}}
$$

and then applies learned scale and shift parameters:

$$
y=\gamma\hat{x}+\beta
$$

PyTorch initializes:

$$
\gamma=1
$$

and:

$$
\beta=0
$$

so initially the LayerNorm output is the same as the normalized values.

During training, $\gamma$ and $\beta$ are learned parameters and can change.

In [73]:
import torch

x = torch.tensor([2.0, 4.0, 6.0, 8.0])

layer_norm = torch.nn.LayerNorm(4)

output = layer_norm(x)

print("Original:", x)
print("LayerNorm output:", output)

print("\nGamma:", layer_norm.weight)
print("Beta:", layer_norm.bias)

print("\nOutput mean:", output.mean())
print("Output variance:", output.var(unbiased=False))

Original: tensor([2., 4., 6., 8.])
LayerNorm output: tensor([-1.3416, -0.4472,  0.4472,  1.3416],
       grad_fn=<NativeLayerNormBackward0>)

Gamma: Parameter containing:
tensor([1., 1., 1., 1.], requires_grad=True)
Beta: Parameter containing:
tensor([0., 0., 0., 0.], requires_grad=True)

Output mean: tensor(0., grad_fn=<MeanBackward0>)
Output variance: tensor(1.0000, grad_fn=<VarBackward0>)


In [74]:
layer_norm = torch.nn.LayerNorm(4)

# Change the learnable parameters
with torch.no_grad():
    layer_norm.weight[:] = 2.0   # gamma
    layer_norm.bias[:] = 1.0     # beta

output = layer_norm(x)

print("Gamma:", layer_norm.weight)
print("Beta:", layer_norm.bias)
print("Output:", output)

Gamma: Parameter containing:
tensor([2., 2., 2., 2.], requires_grad=True)
Beta: Parameter containing:
tensor([1., 1., 1., 1.], requires_grad=True)
Output: tensor([-1.6833,  0.1056,  1.8944,  3.6833],
       grad_fn=<NativeLayerNormBackward0>)


### 1.15.3 Experiment: Attention + Residual Connection + LayerNorm

A Transformer combines these components rather than using attention alone.

First, self-attention produces a new contextual representation:

$$
A=\text{Attention}(X)
$$

The residual connection adds the original representation back:

$$
R=X+A
$$

Then LayerNorm normalizes the resulting representation:

$$
Y=\text{LayerNorm}(R)
$$

Therefore the flow is:

$$
X
\rightarrow
\text{Self-Attention}
\rightarrow
A
$$

while the original $X$ bypasses the attention through the residual connection:

$$
X+A
\rightarrow
\text{LayerNorm}
\rightarrow
Y
$$

This allows the model to preserve the original representation while incorporating information gathered through attention.

In [75]:
import torch

X = torch.tensor([2.0, 4.0, 6.0, 8.0])

attention_output = torch.tensor([1.0, 0.0, 2.0, -1.0])

# Residual connection
R = X + attention_output

# LayerNorm
layer_norm = torch.nn.LayerNorm(4)
Y = layer_norm(R)

print("Original X:", X)
print("Attention output:", attention_output)
print("After residual addition:", R)
print("After LayerNorm:", Y)
print("Mean:", Y.mean())
print("Variance:", Y.var(unbiased=False))

Original X: tensor([2., 4., 6., 8.])
Attention output: tensor([ 1.,  0.,  2., -1.])
After residual addition: tensor([3., 4., 8., 7.])
After LayerNorm: tensor([-1.2127, -0.7276,  1.2127,  0.7276],
       grad_fn=<NativeLayerNormBackward0>)
Mean: tensor(0., grad_fn=<MeanBackward0>)
Variance: tensor(1.0000, grad_fn=<VarBackward0>)


### 1.15.4 Summary: Residual Connections and Layer Normalization

Residual connections and LayerNorm serve different purposes in a Transformer.

#### Residual Connection

A residual connection preserves the existing representation while adding information produced by a transformation.

$$
Y=X+F(X)
$$

For attention:

$$
R=X+\text{Attention}(X)
$$

The original representation therefore has a direct path through the network.

**Primary purpose: information flow and preservation.**

Residual connections:
- preserve information from earlier representations,
- allow a layer to add or modify information rather than completely replace it,
- provide a relatively direct path for gradients during training.

---

#### Layer Normalization

LayerNorm normalizes the feature values within each individual token representation.

For a token representation:

$$
x=[x_1,x_2,\ldots,x_d]
$$

LayerNorm calculates the mean and variance across these $d$ features and normalizes them:

$$
\hat{x}=
\frac{x-\mu}{\sqrt{\sigma^2+\epsilon}}
$$

It then applies learned scale and shift parameters:

$$
y=\gamma\hat{x}+\beta
$$

**Primary purpose: numerical normalization and stability.**

LayerNorm helps keep intermediate representations in a controlled numerical regime as information passes through many Transformer layers.

---

#### How They Work Together

For an attention sublayer, the conceptual flow is:

$$
X
\rightarrow
\text{Self-Attention}
\rightarrow
A
$$

Then the residual connection adds the original representation:

$$
R=X+A
$$

Then LayerNorm normalizes the resulting representation:

$$
Y=\text{LayerNorm}(R)
$$

Therefore:

**Residual connection → information flow and preservation**

**LayerNorm → numerical normalization and stability**

They are complementary rather than interchangeable.

---

#### Key Mental Model

A useful way to think about a Transformer is:

> **Attention gathers information.**

> **Residual connections preserve and carry information forward.**

> **LayerNorm keeps the representation numerically well-behaved.**

These operations will appear repeatedly as we build deeper Transformer blocks.

## 1.16 Feed-Forward Network (FFN)

After self-attention allows tokens to exchange information, the Transformer needs another component to process the resulting representations.

This is the **Feed-Forward Network (FFN)**.

### What does the FFN do?

Self-attention primarily allows **information mixing between tokens**.

For example, the representation of `"pizza"` can gather information from `"I"` and `"love"`.

The FFN then processes each token's resulting representation **independently**.

Therefore:

**Self-Attention → information mixing between tokens**

**FFN → transformation of each token's representation**

The same FFN parameters are applied independently to every token.

---

### Basic FFN Structure

A basic feed-forward network consists of two linear transformations with a nonlinear activation function between them:

$$
h=W_1x+b_1
$$

$$
h=\sigma(W_1x+b_1)
$$

$$
y=W_2h+b_2
$$

Conceptually:

$$
x
\rightarrow
\text{Linear}
\rightarrow
\text{Activation}
\rightarrow
\text{Linear}
\rightarrow
y
$$

---

### Why two linear layers?

The first linear layer generally expands the representation into a larger hidden dimension:

$$
d_{\text{model}}
\rightarrow
d_{\text{ff}}
$$

The second linear layer projects it back:

$$
d_{\text{ff}}
\rightarrow
d_{\text{model}}
$$

For a small example:

$$
4\rightarrow16\rightarrow4
$$

For a real language model, the hidden dimension is typically much larger than the model dimension.

The larger intermediate space gives the network more capacity to perform nonlinear transformations.

---

### Why is the activation function necessary?

If we used only two linear transformations:

$$
y=W_2(W_1x)
$$

their composition would still be equivalent to a single linear transformation:

$$
y=(W_2W_1)x
$$

The activation function introduces **nonlinearity**, allowing the network to learn more complex transformations.

A simple example is ReLU:

$$
\text{ReLU}(x)=\max(0,x)
$$

Modern language models often use other activation mechanisms such as GELU or gated activations such as SwiGLU. These will be studied separately when we examine the actual model architecture.

---

### FFN Operates Independently on Each Token

Suppose the sequence contains four token representations:

$$
X=
\begin{bmatrix}
x_1\\
x_2\\
x_3\\
x_4
\end{bmatrix}
$$

The FFN applies the same learned function to each token:

$$
FFN(x_1),\quad FFN(x_2),\quad FFN(x_3),\quad FFN(x_4)
$$

There is no token-to-token interaction inside the FFN.

The token interaction happens through self-attention.

---

### Transformer Mental Model

The major components studied so far have different roles:

| Component | Main purpose |
|---|---|
| Self-Attention | Mix information between tokens |
| Residual Connection | Preserve and carry information through layers |
| LayerNorm | Normalize representations and improve numerical stability |
| FFN | Transform information within each token |

A simplified Transformer block therefore contains two major computational sublayers:

$$
\text{Self-Attention}
$$

and

$$
\text{FFN}
$$

with residual connections and normalization around these transformations.

The exact ordering depends on the Transformer architecture. Modern decoder-only models commonly use **pre-normalization**, which we will examine when constructing the actual decoder block.

---

### Core Mental Model

**Attention asks:**

> Which other tokens should I use to update this token's representation?

**FFN asks:**

> Given the representation I now have, how should I transform and process it?

Therefore:

$$
\boxed{
\text{Attention = token-to-token information mixing}
}
$$

$$
\boxed{
\text{FFN = per-token nonlinear transformation}
}
$$

### 1.16.1 Experiment: A Tiny Feed-Forward Network

We will first build a minimal FFN without bias or an activation function.

The dimensions are:

$$
d_{\text{input}}=2
$$

$$
d_{\text{hidden}}=4
$$

$$
d_{\text{output}}=2
$$

Therefore the transformation is:

$$
2\rightarrow4\rightarrow2
$$

The first matrix expands the representation:

$$
W_1:2\times4
$$

The second matrix projects it back:

$$
W_2:4\times2
$$

For an input vector $x$:

$$
h=xW_1
$$

and:

$$
y=hW_2
$$

At this stage there is no activation function, so this is only a sequence of linear transformations. We will add the nonlinear activation in the next experiment.

In [76]:
import torch

# Input vector: 2 dimensions
x = torch.tensor([2.0, 4.0])

# First layer: 2 → 4
W1 = torch.tensor([
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 1.0, 0.0, 1.0]
])

# Second layer: 4 → 2
W2 = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
    [0.0, 1.0]
])

# First linear transformation
h = x @ W1

# Second linear transformation
y = h @ W2

print("Input x:")
print(x)
print("Shape:", x.shape)

print("\nAfter first linear layer h:")
print(h)
print("Shape:", h.shape)

print("\nFinal output y:")
print(y)
print("Shape:", y.shape)

Input x:
tensor([2., 4.])
Shape: torch.Size([2])

After first linear layer h:
tensor([2., 4., 2., 4.])
Shape: torch.Size([4])

Final output y:
tensor([ 4., 10.])
Shape: torch.Size([2])


### 1.16.2 Activation Function: ReLU

An activation function introduces nonlinearity into a neural network.

Without an activation function:

$$
y=W_2(W_1x)
$$

which can be simplified to:

$$
y=(W_2W_1)x
$$

Therefore, multiple linear transformations without an activation are still equivalent to one linear transformation.

A nonlinear activation between the two linear layers allows the network to learn more complex transformations.

A simple activation function is ReLU (Rectified Linear Unit):

$$
\text{ReLU}(x)=\max(0,x)
$$

Therefore:

$$
-3\rightarrow0
$$

$$
-1\rightarrow0
$$

$$
2\rightarrow2
$$

$$
5\rightarrow5
$$

The FFN now becomes:

$$
h=\text{ReLU}(xW_1+b_1)
$$

$$
y=hW_2+b_2
$$

The activation is applied element-by-element to the hidden representation.

### 1.16.3 Experiment: ReLU

ReLU operates independently on every element of the hidden representation:

$$
\text{ReLU}(x)=\max(0,x)
$$

For example:

$$
h=[2,-4,3,-1]
$$

Applying ReLU gives:

$$
\text{ReLU}(h)=[2,0,3,0]
$$

Negative values are replaced by zero, while positive values remain unchanged.

In an FFN, ReLU is applied after the first linear transformation and before the second linear transformation:

$$
x
\rightarrow
W_1x+b_1
\rightarrow
\text{ReLU}
\rightarrow
W_2h+b_2
$$

In [77]:
h = torch.tensor([2.0, -4.0, 3.0, -1.0])

relu_output = torch.relu(h)

print("Before ReLU:", h)
print("After ReLU:", relu_output)

Before ReLU: tensor([ 2., -4.,  3., -1.])
After ReLU: tensor([2., 0., 3., 0.])


### 1.16.4 Experiment: ReLU Inside the FFN

We now place ReLU between the two linear transformations.

The FFN is:

$$
h=W_1x+b_1
$$

$$
h=\text{ReLU}(h)
$$

$$
y=W_2h+b_2
$$

The first linear layer can produce both positive and negative values.

ReLU then introduces nonlinearity by replacing negative values with zero.

The second linear layer processes the resulting nonlinear representation.

This is fundamentally different from simply applying two linear transformations one after another.

In [78]:
import torch

x = torch.tensor([2.0, 4.0])

W1 = torch.tensor([
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 1.0, 0.0, 1.0]
])

b1 = torch.tensor([-3.0, 1.0, -5.0, 2.0])

W2 = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
    [0.0, 1.0]
])

b2 = torch.tensor([0.0, 0.0])

# First linear transformation
h_before_activation = x @ W1 + b1

# Apply ReLU
h = torch.relu(h_before_activation)

# Second linear transformation
y = h @ W2 + b2

print("Input:")
print(x)

print("\nBefore ReLU:")
print(h_before_activation)

print("\nAfter ReLU:")
print(h)

print("\nFinal output:")
print(y)

Input:
tensor([2., 4.])

Before ReLU:
tensor([-1.,  5., -3.,  6.])

After ReLU:
tensor([0., 5., 0., 6.])

Final output:
tensor([ 0., 11.])


### 1.16.5 GELU: Moving Toward Modern LLMs

We previously used ReLU as a simple activation function:

$$
\text{ReLU}(x)=\max(0,x)
$$

ReLU has a sharp behavior:

- Negative values become exactly zero.
- Positive values remain unchanged.

For example:

$$
-2\rightarrow0
$$

$$
-1\rightarrow0
$$

$$
1\rightarrow1
$$

$$
2\rightarrow2
$$

---

### GELU

GELU (Gaussian Error Linear Unit) is another nonlinear activation function.

Unlike ReLU, GELU does not make a hard decision to completely remove all negative values.

Instead, it performs a smooth, value-dependent transformation.

Conceptually:

- Very negative values are strongly suppressed.
- Slightly negative values are partially retained.
- Values near zero are smoothly transformed.
- Positive values increasingly pass through.

The exact GELU definition is:

$$
\text{GELU}(x)=x\Phi(x)
$$

where $\Phi(x)$ is the cumulative distribution function of the standard normal distribution.

For now, the important point is that GELU is a smooth nonlinear activation function.

---

### ReLU vs GELU

| Input | ReLU | GELU (approximately) |
|---|---:|---:|
| -2 | 0 | -0.045 |
| -1 | 0 | -0.159 |
| -0.5 | 0 | -0.154 |
| 0 | 0 | 0 |
| 0.5 | 0.5 | 0.346 |
| 1 | 1 | 0.841 |
| 2 | 2 | 1.955 |

Therefore:

**ReLU:** hard cutoff of negative values.

**GELU:** smooth transformation that gradually suppresses or passes values.

---

### GELU in an FFN

Our earlier FFN was:

$$
x
\rightarrow
W_1x+b_1
\rightarrow
\text{ReLU}
\rightarrow
W_2h+b_2
$$

A GELU-based FFN uses:

$$
x
\rightarrow
W_1x+b_1
\rightarrow
\text{GELU}
\rightarrow
W_2h+b_2
$$

The activation function provides the nonlinearity that allows the FFN to learn more complex transformations.

Modern language models may use GELU or other activation mechanisms. Many newer architectures use gated FFNs such as SwiGLU, which we will study next.

### Mathematical Approximation of GELU

The exact GELU function is:

$$
\text{GELU}(x)=x\Phi(x)
$$

A commonly used approximation is:

$$
\text{GELU}(x)
\approx
\frac{x}{2}
\left[
1+
\tanh
\left(
\sqrt{\frac{2}{\pi}}
\left(
x+0.044715x^3
\right)
\right)
\right]
$$

This approximation is computationally efficient and produces values very close to the exact GELU function.

You do not need to memorize this formula.

The important idea is that GELU provides a smooth nonlinear transformation, unlike ReLU which applies a hard cutoff at zero.

### 1.16.6 Experiment: ReLU vs GELU

ReLU and GELU are both nonlinear activation functions, but they behave differently.

ReLU:

$$
\text{ReLU}(x)=\max(0,x)
$$

GELU:

$$
\text{GELU}(x)=x\Phi(x)
$$

ReLU completely removes negative values.

GELU smoothly suppresses negative values and gradually allows positive values to pass through.

We can compare their behavior using the same input values.

In [79]:
x = torch.tensor([-2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0])

relu_output = torch.relu(x)
gelu_output = torch.nn.functional.gelu(x)

print("Input: ", x)
print("ReLU:  ", relu_output)
print("GELU:  ", gelu_output)

Input:  tensor([-2.0000, -1.0000, -0.5000,  0.0000,  0.5000,  1.0000,  2.0000])
ReLU:   tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.5000, 1.0000, 2.0000])
GELU:   tensor([-0.0455, -0.1587, -0.1543,  0.0000,  0.3457,  0.8413,  1.9545])


### 1.16.7 Experiment: GELU Inside the FFN

We now use GELU inside the same FFN that previously used ReLU.

The architecture remains:

$$
2\rightarrow4\rightarrow2
$$

The FFN calculation is:

$$
h_{\text{before activation}}=xW_1+b_1
$$

Then:

$$
h=\text{GELU}(h_{\text{before activation}})
$$

Finally:

$$
y=hW_2+b_2
$$

We use the same input and weight matrices as the ReLU experiment.

Therefore, any difference in the final output comes from the difference between the activation functions.

The flow is:

$$
\text{Input}
\rightarrow
\text{Linear}
\rightarrow
\text{GELU}
\rightarrow
\text{Linear}
\rightarrow
\text{Output}
$$

In [80]:
import torch
import torch.nn.functional as F

x = torch.tensor([2.0, 4.0])

W1 = torch.tensor([
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 1.0, 0.0, 1.0]
])

b1 = torch.tensor([-3.0, 1.0, -5.0, 2.0])

W2 = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
    [0.0, 1.0]
])

b2 = torch.tensor([0.0, 0.0])

# First linear transformation
h_before_activation = x @ W1 + b1

# ReLU version
h_relu = torch.relu(h_before_activation)
y_relu = h_relu @ W2 + b2

# GELU version
h_gelu = F.gelu(h_before_activation)
y_gelu = h_gelu @ W2 + b2

print("Before activation:")
print(h_before_activation)

print("\nAfter ReLU:")
print(h_relu)

print("\nAfter GELU:")
print(h_gelu)

print("\nFinal output with ReLU:")
print(y_relu)

print("\nFinal output with GELU:")
print(y_gelu)

Before activation:
tensor([-1.,  5., -3.,  6.])

After ReLU:
tensor([0., 5., 0., 6.])

After GELU:
tensor([-1.5866e-01,  5.0000e+00, -4.0497e-03,  6.0000e+00])

Final output with ReLU:
tensor([ 0., 11.])

Final output with GELU:
tensor([-0.1627, 10.9959])


### 1.16.9 Experiment: Understanding Sigmoid

The sigmoid function transforms any input value into a value between 0 and 1.

Its formula is:

$$
\sigma(x)
=
\frac{1}{1+e^{-x}}
$$

The output always lies between:

$$
0 < \sigma(x) < 1
$$

Key behavior:

- Large negative values → close to 0
- Negative values → below 0.5
- Zero → exactly 0.5
- Positive values → above 0.5
- Large positive values → close to 1

Sigmoid is applied independently to every element of a vector.

For example:

$$
[-2,-1,0,1,2]
$$

is transformed approximately into:

$$
[0.119,0.269,0.5,0.731,0.881]
$$

We will now verify this using PyTorch.

In [81]:
import torch

x = torch.tensor([-5.0, -2.0, -1.0, 0.0, 1.0, 2.0, 5.0])

sigmoid_output = torch.sigmoid(x)

print("Input:")
print(x)

print("\nSigmoid output:")
print(sigmoid_output)

Input:
tensor([-5., -2., -1.,  0.,  1.,  2.,  5.])

Sigmoid output:
tensor([0.0067, 0.1192, 0.2689, 0.5000, 0.7311, 0.8808, 0.9933])


### 1.16.10 Experiment: Understanding SiLU

SiLU is an activation function.

Its formula is:

$$
\text{SiLU}(x)
=
x\cdot\sigma(x)
$$

where:

$$
\sigma(x)
=
\frac{1}{1+e^{-x}}
$$

Therefore, SiLU uses two components:

1. The original input:

$$
x
$$

2. A sigmoid-based scaling factor:

$$
\sigma(x)
$$

These are multiplied element by element:

$$
\text{SiLU}(x)
=
x\odot\sigma(x)
$$

For a single value:

$$
x=2
$$

First:

$$
\sigma(2)\approx0.8808
$$

Then:

$$
\text{SiLU}(2)
=
2\times0.8808
$$

$$
\approx1.7616
$$

We will now calculate SiLU manually using sigmoid and compare it with PyTorch's built-in SiLU function.

In [82]:
x = torch.tensor([-5.0, -2.0, -1.0, 0.0, 1.0, 2.0, 5.0])

# Step 1: Calculate sigmoid
sigmoid_x = torch.sigmoid(x)

# Step 2: Manually calculate SiLU
silu_manual = x * sigmoid_x

# Step 3: PyTorch built-in SiLU
silu_builtin = torch.nn.functional.silu(x)

print("Input:")
print(x)

print("\nSigmoid:")
print(sigmoid_x)

print("\nManual SiLU = x * sigmoid(x):")
print(silu_manual)

print("\nPyTorch SiLU:")
print(silu_builtin)

print("\nAre they equal?")
print(torch.allclose(silu_manual, silu_builtin))

Input:
tensor([-5., -2., -1.,  0.,  1.,  2.,  5.])

Sigmoid:
tensor([0.0067, 0.1192, 0.2689, 0.5000, 0.7311, 0.8808, 0.9933])

Manual SiLU = x * sigmoid(x):
tensor([-0.0335, -0.2384, -0.2689,  0.0000,  0.7311,  1.7616,  4.9665])

PyTorch SiLU:
tensor([-0.0335, -0.2384, -0.2689,  0.0000,  0.7311,  1.7616,  4.9665])

Are they equal?
True


### 1.16.11 Experiment: Building a Complete SwiGLU FFN

We will now build a complete SwiGLU FFN using a small example.

Our input representation is:

$$
x=[2,4]
$$

The model dimension is:

$$
d_{\text{model}}=2
$$

We expand to:

$$
d_{\text{hidden}}=4
$$

SwiGLU uses three learned projections:

$$
W_{\text{up}}
$$

creates the information representation:

$$
\text{Information}=xW_{\text{up}}
$$

$$
W_{\text{gate}}
$$

creates the gate representation:

$$
\text{Gate raw}=xW_{\text{gate}}
$$

The gate representation then passes through SiLU:

$$
\text{Gate}
=
\text{SiLU}(\text{Gate raw})
$$

The information and gate representations are combined using element-wise multiplication:

$$
\text{Hidden}
=
\text{Information}
\odot
\text{Gate}
$$

Finally:

$$
\text{Output}
=
\text{Hidden}W_{\text{down}}
$$

The complete SwiGLU calculation is:

$$
\boxed{
\text{Output}
=
\left(
xW_{\text{up}}
\odot
\text{SiLU}(xW_{\text{gate}})
\right)
W_{\text{down}}
}
$$

In [83]:
x = torch.tensor([2.0, 4.0])

W_up = torch.tensor([
    [1.0, 0.0, 0.5, 1.0],
    [0.0, 1.0, 1.0, 0.5]
])

W_gate = torch.tensor([
    [0.5, 1.0, -1.0, 0.5],
    [1.0, -0.5, 0.5, 1.0]
])

W_down = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [0.5, 0.5],
    [0.2, 0.3]
])

print("Input x:")
print(x)
print("Shape:", x.shape)

print("\nW_up:")
print(W_up)
print("Shape:", W_up.shape)

print("\nW_gate:")
print(W_gate)
print("Shape:", W_gate.shape)

print("\nW_down:")
print(W_down)
print("Shape:", W_down.shape)

Input x:
tensor([2., 4.])
Shape: torch.Size([2])

W_up:
tensor([[1.0000, 0.0000, 0.5000, 1.0000],
        [0.0000, 1.0000, 1.0000, 0.5000]])
Shape: torch.Size([2, 4])

W_gate:
tensor([[ 0.5000,  1.0000, -1.0000,  0.5000],
        [ 1.0000, -0.5000,  0.5000,  1.0000]])
Shape: torch.Size([2, 4])

W_down:
tensor([[1.0000, 0.0000],
        [0.0000, 1.0000],
        [0.5000, 0.5000],
        [0.2000, 0.3000]])
Shape: torch.Size([4, 2])


In [91]:
((x @ W_up) * torch.nn.functional.silu(x @ W_gate)) @ W_down

tensor([13.9063,  5.9598])

### 1.16.12 Calculate the Two Parallel Projections

The same input representation:

$$
x=[2,4]
$$

is sent through two different learned linear projections.

### Up Projection

The information path is:

$$
\text{Information}
=
xW_{\text{up}}
$$

This produces a 4-dimensional representation:

$$
[2]\times[2,4]
\rightarrow
[4]
$$

### Gate Projection

The gate path is:

$$
\text{Gate raw}
=
xW_{\text{gate}}
$$

This also produces a 4-dimensional representation:

$$
[2]\times[2,4]
\rightarrow
[4]
$$

Therefore:

```text
Input x = [2, 4]

        ┌──────── W_up ────────→ Information [4]
        │
        │
        └──────── W_gate ──────→ Gate raw [4]

In [92]:

# Now run:

# ```python
# Up / information projection
information = x @ W_up

# Gate projection
gate_raw = x @ W_gate

print("Input x:")
print(x)

print("\nInformation = x @ W_up:")
print(information)
print("Shape:", information.shape)

print("\nGate raw = x @ W_gate:")
print(gate_raw)
print("Shape:", gate_raw.shape)

Input x:
tensor([2., 4.])

Information = x @ W_up:
tensor([2., 4., 5., 4.])
Shape: torch.Size([4])

Gate raw = x @ W_gate:
tensor([5., 0., 0., 5.])
Shape: torch.Size([4])


In [93]:
information = x @ W_up

gate_raw = x @ W_gate

gate = torch.nn.functional.silu(gate_raw)

hidden = information * gate

output = hidden @ W_down

print("1. Information / Up path:")
print(information)

print("\n2. Gate raw:")
print(gate_raw)

print("\n3. Gate after SiLU:")
print(gate)

print("\n4. Information × Gate:")
print(hidden)

print("\n5. Final output:")
print(output)

1. Information / Up path:
tensor([2., 4., 5., 4.])

2. Gate raw:
tensor([5., 0., 0., 5.])

3. Gate after SiLU:
tensor([4.9665, 0.0000, 0.0000, 4.9665])

4. Information × Gate:
tensor([ 9.9331,  0.0000,  0.0000, 19.8661])

5. Final output:
tensor([13.9063,  5.9598])


In [ ]:
import torch
import torch.nn.functional as F

x = torch.tensor([2.0, 4.0])

W1 = torch.tensor([
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 1.0, 0.0, 1.0]
])

b1 = torch.tensor([-3.0, 1.0, -5.0, 2.0])

W2 = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
    [0.0, 1.0]
])

b2 = torch.tensor([0.0, 0.0])

W_up = torch.tensor([
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 1.0, 0.0, 1.0]
])

W_gate = torch.tensor([
    [0.5, 1.0, -1.0, 0.5],
    [1.0, -0.5, 0.5, 1.0]
])

W_down = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [0.5, 0.5],
    [0.2, 0.3]
])

# gelu
print(F.gelu((x @ W1) + b1) @ W2 + b2)
print(((x @ W_up) * F.silu(x @ W_gate)) @ W_down)
print(((x @ W_up) + b1) * (F.silu(x @ W_gate + b1)) @ W_down + b2) # just for experiment - if we keep bias then b_up and b_gate to be different and not same.

tensor([-0.1627, 10.9959])
tensor([13.9063,  5.9598])
tensor([ 6.6810, 16.2940])


# 1.17 Gated Feed Forward Networks and SwiGLU

## 1.17.1 Why Do We Need a Gated FFN?

Previously, we learned the standard Transformer Feed Forward Network (FFN).

Its architecture is:

    Input
      ↓
    Linear / Expansion
      ↓
    Activation
      ↓
    Linear / Down Projection
      ↓
    Output

Mathematically:

$$
\text{FFN}(x)
=
\text{Activation}(xW_1+b_1)W_2+b_2
$$

For example, using GELU:

$$
\text{FFN}(x)
=
\text{GELU}(xW_1+b_1)W_2+b_2
$$

The standard FFN creates one hidden representation and applies a nonlinear transformation to it.

Modern Transformer architectures often use a more expressive mechanism called a **Gated Feed Forward Network**.

Instead of creating only one hidden representation, the input is sent through two different learned projections:

1. An information path
2. A gate path

The gate path learns how to modulate the information path.

Conceptually:

    Input
       │
       ├──────────────→ Information
       │                   │
       │                   ×
       │                   │
       └──────────────→ Gate
                           │
                        Activation
                           │
                           ↓

This makes the FFN more flexible.

---

## 1.17.2 Understanding Sigmoid

Before understanding SwiGLU, we first need to understand the sigmoid function.

Sigmoid transforms any input value into a value between 0 and 1.

Its formula is:

$$
\boxed{
\sigma(x)
=
\frac{1}{1+e^{-x}}
}
$$

The input can range from:

$$
-\infty
\text{ to }
+\infty
$$

but the output is always between:

$$
0
\text{ and }
1
$$

Conceptually:

    Very negative input
            ↓
          Near 0


    Input = 0
            ↓
          0.5


    Very positive input
            ↓
          Near 1

For example:

| Input | Sigmoid Output |
|---|---:|
| -5 | 0.0067 |
| -2 | 0.1192 |
| -1 | 0.2689 |
| 0 | 0.5000 |
| 1 | 0.7311 |
| 2 | 0.8808 |
| 5 | 0.9933 |

---

## 1.17.3 Why is Sigmoid(0) = 0.5?

Using the formula:

$$
\sigma(x)
=
\frac{1}{1+e^{-x}}
$$

For:

$$
x=0
$$

we get:

$$
\sigma(0)
=
\frac{1}{1+e^0}
$$

Since:

$$
e^0=1
$$

therefore:

$$
\sigma(0)
=
\frac{1}{1+1}
=
0.5
$$

So zero sits exactly in the middle of the sigmoid curve.

---

## 1.17.4 Sigmoid Intuition

Sigmoid can be thought of as producing a smooth scaling value.

Suppose we have information:

$$
[10,20,30]
$$

and scaling values:

$$
[0.1,0.8,0.3]
$$

Element-wise multiplication gives:

$$
[10,20,30]
\odot
[0.1,0.8,0.3]
$$

which becomes:

$$
[1,16,9]
$$

Conceptually:

    Value near 0
        ↓
    Strong suppression


    Value near 1
        ↓
    Mostly retain

This gives us the basic intuition behind gating.

However, SwiGLU does not directly use sigmoid as its gate output.

It uses SiLU.

---

# 1.17.5 SiLU Activation Function

SiLU stands for:

**Sigmoid Linear Unit**

Its formula is:

$$
\boxed{
\text{SiLU}(x)
=
x\cdot\sigma(x)
}
$$

Since:

$$
\sigma(x)
=
\frac{1}{1+e^{-x}}
$$

we can also write:

$$
\boxed{
\text{SiLU}(x)
=
x
\left(
\frac{1}{1+e^{-x}}
\right)
}
$$

SiLU takes the original input and multiplies it by its sigmoid value.

Conceptually:

    Input x
       │
       ├───────────────┐
       │               │
       │            Sigmoid
       │               ↓
       │             σ(x)
       │               │
       └───────────────×
                       │
                       ↓
                     SiLU(x)

---

## 1.17.6 SiLU Numerical Example

Consider:

$$
x=2
$$

First calculate sigmoid:

$$
\sigma(2)
\approx
0.8808
$$

Then:

$$
\text{SiLU}(2)
=
2\times0.8808
$$

Therefore:

$$
\boxed{
\text{SiLU}(2)
\approx1.7616
}
$$

Now consider:

$$
x=-1
$$

First:

$$
\sigma(-1)
\approx0.2689
$$

Then:

$$
\text{SiLU}(-1)
=
-1\times0.2689
$$

Therefore:

$$
\boxed{
\text{SiLU}(-1)
=
-0.2689
}
$$

Some example values:

| Input | Sigmoid | SiLU |
|---|---:|---:|
| -5 | 0.0067 | -0.0335 |
| -2 | 0.1192 | -0.2384 |
| -1 | 0.2689 | -0.2689 |
| 0 | 0.5000 | 0 |
| 1 | 0.7311 | 0.7311 |
| 2 | 0.8808 | 1.7616 |
| 5 | 0.9933 | 4.9665 |

---

## 1.17.7 SiLU vs ReLU

ReLU is defined as:

$$
\text{ReLU}(x)
=
\max(0,x)
$$

Therefore:

    Negative input
          ↓
      Hard cutoff
          ↓
          0

For example:

$$
\text{ReLU}(-2)=0
$$

SiLU behaves differently.

For:

$$
x=-2
$$

we get:

$$
\text{SiLU}(-2)
\approx-0.2384
$$

Therefore, SiLU does not completely remove negative values.

Instead, it smoothly suppresses them.

| Input | ReLU | SiLU |
|---|---:|---:|
| -2 | 0 | -0.2384 |
| -1 | 0 | -0.2689 |
| 0 | 0 | 0 |
| 1 | 1 | 0.7311 |
| 2 | 2 | 1.7616 |

Conceptually:

    ReLU

    Negative
       ↓
    Hard cutoff
       ↓
       0


    SiLU

    Negative
       ↓
    Smooth suppression
       ↓
    Small negative value

---

# 1.17.8 Introduction to SwiGLU

SwiGLU is a gated feed forward mechanism.

A standard FFN looks like:

    Input
      ↓
    Linear
      ↓
    Activation
      ↓
    Linear
      ↓
    Output

SwiGLU uses two parallel learned projections:

    Input

        ┌──────── W_up ────────→ Information
        │                           │
        │                           │
        │                           ×
        │                           │
        │                           │
        └────── W_gate ──→ SiLU ───┘
                                    │
                                    ↓
                               W_down
                                    │
                                    ↓
                                  Output

The same input is transformed in two different ways.

---

## 1.17.9 The Two Parallel Projections

Let the input representation be:

$$
x
$$

SwiGLU creates two representations.

### Information Path

The first projection is:

$$
\boxed{
\text{Information}
=
xW_{\text{up}}
}
$$

This creates a learned hidden representation.

### Gate Path

The second projection is:

$$
\text{Gate raw}
=
xW_{\text{gate}}
$$

The result then passes through SiLU:

$$
\boxed{
\text{Gate}
=
\text{SiLU}
(
xW_{\text{gate}}
)
}
$$

Conceptually:

    Input x

            ┌──────── W_up ─────────→ Information
            │
            │
            └──────── W_gate ───────→ Gate raw
                                          │
                                        SiLU
                                          │
                                          ↓
                                         Gate

The two paths use different learned weight matrices:

$$
\boxed{
W_{\text{up}}
\neq
W_{\text{gate}}
}
$$

in general.

This is important.

The model is learning two different transformations of the same input.

---

## 1.17.10 Why Do We Call One Path "Information"?

The names **information path** and **gate path** are conceptual names that help us understand the architecture.

The up projection:

$$
xW_{\text{up}}
$$

creates a learned hidden representation.

For example:

$$
[2,4]
\rightarrow
[2,4,5,4]
$$

These numbers are learned features.

The model does not explicitly label them as:

    Feature 1 = grammar
    Feature 2 = subject
    Feature 3 = sentiment
    Feature 4 = meaning

Instead, during training, the model learns useful numerical features.

Therefore:

> The up projection creates the information or feature representation that will be passed forward.

---

## 1.17.11 Why Do We Call the Other Path a Gate?

The gate path also creates a learned representation:

$$
xW_{\text{gate}}
$$

Then SiLU transforms it:

$$
\text{SiLU}(xW_{\text{gate}})
$$

This result is multiplied element by element with the information representation.

Therefore, it directly controls how strongly each information dimension contributes.

Suppose:

$$
\text{Information}
=
[2,4,5,4]
$$

and:

$$
\text{Gate}
=
[4.9,0,0,4.9]
$$

Then:

$$
[2,4,5,4]
\odot
[4.9,0,0,4.9]
$$

becomes approximately:

$$
[9.8,0,0,19.6]
$$

Therefore:

    Information dimension 1
        ↓
    Amplified


    Information dimension 2
        ↓
    Suppressed


    Information dimension 3
        ↓
    Suppressed


    Information dimension 4
        ↓
    Amplified

This is why we call it a gate.

---

## 1.17.12 Combining Information and Gate

The two representations are combined using element-wise multiplication.

$$
\boxed{
\text{Hidden}
=
\text{Information}
\odot
\text{Gate}
}
$$

Substituting the two projections:

$$
\boxed{
\text{Hidden}
=
xW_{\text{up}}
\odot
\text{SiLU}(xW_{\text{gate}})
}
$$

Element-wise multiplication means:

$$
[a,b,c,d]
\odot
[w,x,y,z]
$$

becomes:

$$
[a\cdot w,b\cdot x,c\cdot y,d\cdot z]
$$

Both vectors must have the same dimension.

For example:

$$
[4]
\odot
[4]
\rightarrow
[4]
$$

This is different from matrix multiplication.

### Element-wise multiplication

$$
[a,b,c]
\odot
[x,y,z]
=
[ax,by,cz]
$$

SwiGLU specifically uses element-wise multiplication between the information and gate paths.

---

## 1.17.13 Complete SwiGLU Formula

After combining information and gate, the hidden representation is projected back to the model dimension.

$$
\text{Output}
=
\text{Hidden}
W_{\text{down}}
$$

Substituting the complete hidden representation:

$$
\boxed{
\text{Output}
=
\left(
xW_{\text{up}}
\odot
\text{SiLU}(xW_{\text{gate}})
\right)
W_{\text{down}}
}
$$

This is the complete bias-free SwiGLU formula.

The full flow is:

    Input x

        │

        ├────────── W_up ──────────→ Information

        │                                │

        │                                ×

        │                                │

        └──────── W_gate ─→ SiLU ───────┘

                                         │

                                         ↓

                                      Hidden

                                         │

                                      W_down

                                         │

                                         ↓

                                       Output

---

## 1.17.14 Complete Dimension Flow

Suppose:

$$
d_{\text{model}}=2
$$

and:

$$
d_{\text{hidden}}=4
$$

The input is:

    [2]

Both projections expand it:

    Input
    [2]

            ┌──────── W_up ─────────→ [4]
            │                           Information
            │
            │
            └──────── W_gate ───────→ [4]
                                         ↓
                                       SiLU
                                         ↓
                                        [4]
                                         Gate

The two vectors are multiplied:

$$
[4]
\odot
[4]
\rightarrow
[4]
$$

Finally:

    Hidden
    [4]
     │
     │ W_down
     ↓
    [2]

    Output

Therefore:

    [2]

     ↓

    Two parallel projections

     ↓

    [4] Information
           ×
    [4] Gate

     ↓

    [4] Hidden

     ↓

    [2] Output

---

## 1.17.15 Complete Numerical SwiGLU Example

Consider:

$$
x=[2,4]
$$

We use:

$$
W_{\text{up}}
=
\begin{bmatrix}
1 & 0 & 1 & 0 \\
0 & 1 & 0 & 1
\end{bmatrix}
$$

and:

$$
W_{\text{gate}}
=
\begin{bmatrix}
0.5 & 1 & -1 & 0.5 \\
1 & -0.5 & 0.5 & 1
\end{bmatrix}
$$

and:

$$
W_{\text{down}}
=
\begin{bmatrix}
1 & 0 \\
0 & 1 \\
0.5 & 0.5 \\
0.2 & 0.3
\end{bmatrix}
$$

### Step 1: Information Path

Calculate:

$$
xW_{\text{up}}
$$

Result:

$$
\boxed{
[2,4,5,4]
}
$$

This is the information representation.

### Step 2: Gate Path

Calculate:

$$
xW_{\text{gate}}
$$

Result:

$$
\boxed{
[5,0,0,5]
}
$$

This is the raw gate representation.

### Step 3: Apply SiLU

Calculate:

$$
\text{SiLU}
(
[5,0,0,5]
)
$$

Result:

$$
\boxed{
[4.9665,0,0,4.9665]
}
$$

This is the gate representation.

### Step 4: Element-wise Multiplication

Information:

$$
[2,4,5,4]
$$

Gate:

$$
[4.9665,0,0,4.9665]
$$

Multiply element by element:

$$
[2\times4.9665,
4\times0,
5\times0,
4\times4.9665]
$$

Result:

$$
\boxed{
[9.9331,0,0,19.8661]
}
$$

This is the hidden representation.

Notice what happened:

    Information

    [2,       4,       5,       4]

    Gate

    [4.9665,  0,       0,       4.9665]

                 ↓

    Hidden

    [9.9331,  0,       0,       19.8661]

The second and third dimensions were strongly suppressed.

The first and fourth dimensions were amplified.

### Step 5: Down Projection

The hidden representation is:

$$
[9.9331,0,0,19.8661]
$$

This is projected back to the model dimension using:

$$
W_{\text{down}}
$$

The result is:

$$
\boxed{
[13.9063,5.9598]
}
$$

Therefore, the complete flow was:

    Input

    [2,4]

        │

        ├──── W_up ────→ [2,4,5,4]

        │

        └─── W_gate ───→ [5,0,0,5]

                             │

                           SiLU

                             ↓

                      [4.9665,0,0,4.9665]

                             │

                             ×

                             │

                      [9.9331,0,0,19.8661]

                             │

                          W_down

                             ↓

                      [13.9063,5.9598]

---

## 1.17.16 Important Gate Intuition

It is useful to think of a gate as controlling information.

However, SwiGLU is not a simple binary gate.

It is not simply:

    0 = OFF

    1 = ON

Because SiLU can produce values:

- Near zero
- Between zero and one
- Greater than one
- Small negative values

Therefore, the gate can:

    Near 0
       ↓
    Strong suppression


    Between 0 and 1
       ↓
    Reduction


    Around 1
       ↓
    Approximate retention


    Greater than 1
       ↓
    Amplification

For example:

$$
\text{SiLU}(5)
=
4.9665
$$

Therefore:

$$
2\times4.9665
=
9.933
$$

The information is amplified.

Therefore, a better interpretation is:

> The gate learns how to modulate each hidden feature.

rather than:

> The gate simply decides ON or OFF.

---

## 1.17.17 Standard FFN vs SwiGLU

### Standard FFN

The architecture is:

    Input
      ↓
    Linear / Expansion
      ↓
    Activation
      ↓
    Linear / Down Projection
      ↓
    Output

Formula:

$$
\boxed{
\text{Output}
=
\text{GELU}(xW_1)W_2
}
$$

Or, when biases are used:

$$
\text{Output}
=
\text{GELU}(xW_1+b_1)W_2+b_2
$$

There is one hidden representation path.

### SwiGLU

The architecture is:

    Input

            ┌──────────→ Information
            │                │
            │                ×
            │                │
            └──────────→ Gate
                            │
                          SiLU
                            │

                             ↓

                       Down Projection

                             ↓

                           Output

Formula:

$$
\boxed{
\text{Output}
=
\left(
xW_{\text{up}}
\odot
\text{SiLU}(xW_{\text{gate}})
\right)
W_{\text{down}}
}
$$

There are two parallel hidden representation paths.

### Comparison

| Feature | Standard FFN | SwiGLU |
|---|---|---|
| Input paths | One | Two |
| First projection | $W_1$ | $W_{\text{up}}$ |
| Additional projection | None | $W_{\text{gate}}$ |
| Activation | GELU / ReLU | SiLU on gate path |
| Combination | Direct activation | Element-wise multiplication |
| Output projection | $W_2$ | $W_{\text{down}}$ |

The fundamental difference is:

### Standard FFN

    Input
     ↓
    Create representation
     ↓
    Apply nonlinearity
     ↓
    Project back

### SwiGLU

    Input
     ↓
    Create TWO learned representations

    Information

    Gate
     ↓
    SiLU

    Information × Gate
     ↓

    Project back

---

## 1.17.18 Biases in SwiGLU

Biases can theoretically be used in SwiGLU.

The information path would become:

$$
\text{Information}
=
xW_{\text{up}}
+
b_{\text{up}}
$$

The gate path would become:

$$
\text{Gate}
=
\text{SiLU}
(
xW_{\text{gate}}
+
b_{\text{gate}}
)
$$

The final output would become:

$$
\boxed{
\text{Output}
=
\left(
(xW_{\text{up}}+b_{\text{up}})
\odot
\text{SiLU}
(
xW_{\text{gate}}+b_{\text{gate}}
)
\right)
W_{\text{down}}
+
b_{\text{down}}
}
$$

The important point is that the two paths have separate biases:

$$
\boxed{
b_{\text{up}}
\neq
b_{\text{gate}}
}
$$

in general.

This is because they represent different learned projections.

Conceptually:

    Information path

    xW_up
      +
    b_up


    Gate path

    xW_gate
      +
    b_gate
      ↓
    SiLU

---

## 1.17.19 Bias-Free SwiGLU

Many modern Transformer architectures use bias-free projections.

The formula then becomes:

$$
\boxed{
\text{Output}
=
\left(
xW_{\text{up}}
\odot
\text{SiLU}(xW_{\text{gate}})
\right)
W_{\text{down}}
}
$$

Our PyTorch implementation is:

    output = (
        (x @ W_up)
        *
        F.silu(x @ W_gate)
    ) @ W_down

This directly maps to the mathematical formula.

---

## 1.17.20 SwiGLU in SmolLM2

SmolLM2 uses a SwiGLU-style MLP.

It contains three projections:

    gate_proj

    up_proj

    down_proj

The architecture is:

    Input x

            ┌──────────── gate_proj ───────────→ SiLU
            │                                      │
            │                                      │
            │                                      ×
            │                                      │
            └──────────── up_proj ─────────────────┘
                                                   │
                                                   ↓

                                              down_proj

                                                   ↓

                                                 Output

Conceptually:

$$
\boxed{
\text{Output}
=
\text{down\_proj}
\left(
\text{SiLU}
(
\text{gate\_proj}(x)
)
\odot
\text{up\_proj}(x)
\right)
}
$$

SmolLM2 uses bias-free MLP projections:

    gate_proj
    bias = False


    up_proj
    bias = False


    down_proj
    bias = False

The implementation can be understood as:

    gate = F.silu(gate_proj(x))

    up = up_proj(x)

    hidden = gate * up

    output = down_proj(hidden)

Or more compactly:

    output = down_proj(
        F.silu(gate_proj(x))
        *
        up_proj(x)
    )

Therefore, the simplified SwiGLU implementation we built manually matches the core structure used in SmolLM2.

---

## 1.17.21 Final Mental Model

The entire SwiGLU mechanism can be remembered as:

    Input

       │

       ├──────────────→ Create Information

       │

       └──────────────→ Create Gate
                              │
                            SiLU
                              │
                              ↓

                     Modulate Information

                              │

                     Element-wise Multiply

                              │

                       Project Back

                              │

                            Output

Mathematically:

$$
\boxed{
\text{Output}
=
\left(
xW_{\text{up}}
\odot
\text{SiLU}(xW_{\text{gate}})
\right)
W_{\text{down}}
}
$$

The core intuition is:

> The up projection creates a learned information representation.

> The gate projection creates another learned representation that determines how the information should be modulated.

> SiLU transforms the gate representation.

> Element-wise multiplication combines information and gate.

> The down projection brings the hidden representation back to the model dimension.

---

# Summary

SwiGLU consists of five main steps:

$$
\boxed{
\begin{aligned}
\text{Information}
&=
xW_{\text{up}}
\\
\\
\text{Gate}
&=
\text{SiLU}(xW_{\text{gate}})
\\
\\
\text{Hidden}
&=
\text{Information}
\odot
\text{Gate}
\\
\\
\text{Output}
&=
\text{Hidden}W_{\text{down}}
\end{aligned}
}
$$

Or in one formula:

$$
\boxed{
\text{SwiGLU}(x)
=
\left(
xW_{\text{up}}
\odot
\text{SiLU}(xW_{\text{gate}})
\right)
W_{\text{down}}
}
$$

# 1.18 RMSNorm

## 1.18.1 What is RMSNorm?

RMSNorm stands for:

> Root Mean Square Normalization

It is a normalization technique commonly used in modern Transformer and LLM architectures.

Previously, we learned LayerNorm.

### LayerNorm

LayerNorm performs roughly:

    Input
      ↓
    Calculate Mean
      ↓
    Subtract Mean
      ↓
    Calculate Variance
      ↓
    Normalize
      ↓
    Scale and Shift

RMSNorm is simpler:

    Input
      ↓
    Calculate RMS
      ↓
    Divide by RMS
      ↓
    Scale with Gamma

The biggest difference is:

> RMSNorm does not subtract the mean.

LayerNorm centers the representation around zero before normalizing.

RMSNorm keeps the representation centered where it is and only controls its overall magnitude.

---

## 1.18.2 What is RMS?

RMS means:

> Root Mean Square

The calculation happens in three steps:

    Values
      ↓
    Square
      ↓
    Mean
      ↓
    Square Root
      ↓
    RMS

For a vector:

$$
x=[x_1,x_2,\dots,x_d]
$$

the RMS is:

$$
\boxed{
RMS(x)
=
\sqrt{
\frac{1}{d}
\sum_{i=1}^{d}x_i^2
}
}
$$

### What does RMS tell us?

RMS gives us a measure of the overall magnitude of a vector.

For example:

    [0.1, 0.2, 0.1, 0.3]

        ↓

    Small overall magnitude


    [10, 20, 15, 25]

        ↓

    Large overall magnitude

RMSNorm uses this overall magnitude to scale the representation.

---

## 1.18.3 RMS Calculation Example

Consider:

$$
x=[3,4]
$$

### Step 1: Square Each Value

$$
[3^2,4^2]
$$

$$
=
[9,16]
$$

### Step 2: Calculate the Mean

$$
\frac{9+16}{2}
$$

$$
=
12.5
$$

### Step 3: Take the Square Root

$$
\sqrt{12.5}
$$

$$
\approx3.5355
$$

Therefore:

$$
\boxed{
RMS([3,4])
\approx3.5355
}
$$

The name Root Mean Square comes directly from the calculation:

1. Square the values
2. Calculate their Mean
3. Take the Square Root

Therefore:

$$
\boxed{
RMS
=
Root(Mean(Square(x)))
}
$$

---

## 1.18.4 Basic RMS Normalization

Once we calculate the RMS, we divide every value by it.

The basic normalization formula is:

$$
\boxed{
\text{Normalized }x
=
\frac{x}{RMS(x)}
}
$$

Let's take:

$$
x=[2,4,6,8]
$$

### Step 1: Square Every Value

$$
[2^2,4^2,6^2,8^2]
$$

$$
=
[4,16,36,64]
$$

### Step 2: Calculate the Mean of Squares

$$
\frac{4+16+36+64}{4}
$$

$$
=
\frac{120}{4}
$$

$$
=
30
$$

### Step 3: Calculate RMS

$$
RMS(x)
=
\sqrt{30}
$$

$$
\boxed{
RMS(x)
\approx5.4772
}
$$

### Step 4: Divide Every Value by RMS

$$
\frac{[2,4,6,8]}{5.4772}
$$

Result:

$$
\boxed{
[0.3651,0.7303,1.0954,1.4606]
}
$$

This is the basic RMS-normalized representation.

---

## 1.18.5 RMSNorm vs LayerNorm

Let's compare both techniques using the same input:

$$
x=[2,4,6,8]
$$

### LayerNorm

First calculate the mean:

$$
\mu
=
\frac{2+4+6+8}{4}
$$

$$
=
5
$$

Subtract the mean:

$$
[2,4,6,8]-5
$$

$$
=
[-3,-1,1,3]
$$

Then LayerNorm normalizes the variance.

The final normalized output is:

$$
\boxed{
[-1.3416,-0.4472,0.4472,1.3416]
}
$$

LayerNorm performs:

$$
\boxed{
x
\rightarrow
x-\mu
\rightarrow
\text{Normalize}
}
$$

Therefore, LayerNorm centers the representation around zero.

---

### RMSNorm

RMSNorm does not calculate or subtract the mean.

It directly calculates RMS:

$$
RMS(x)
=
\sqrt{
\frac{1}{d}
\sum x_i^2
}
$$

Then divides the input by RMS:

$$
\frac{x}{RMS(x)}
$$

For our example:

$$
\boxed{
[0.3651,0.7303,1.0954,1.4606]
}
$$

Therefore:

    LayerNorm

    "Center the representation
    and normalize its spread."


    RMSNorm

    "Keep the representation direction
    but control its overall magnitude."

---

### Important Difference

LayerNorm forces the normalized representation to have:

$$
\boxed{
Mean\approx0
}
$$

and:

$$
\boxed{
Variance\approx1
}
$$

before learnable scale and shift are applied.

RMSNorm does not force the mean to zero.

For example, the RMS-normalized output:

$$
[0.3651,0.7303,1.0954,1.4606]
$$

has mean approximately:

$$
0.9129
$$

Therefore:

$$
\boxed{
RMSNorm
\neq
Zero Mean Normalization
}
$$

Instead, RMSNorm ensures:

$$
\boxed{
RMS(\text{output})
\approx1
}
$$

---

## 1.18.6 Epsilon and Gamma

The actual RMSNorm implementation includes two additional components:

1. Epsilon
2. Gamma

---

### Epsilon

A small value called epsilon is added for numerical stability.

It is represented as:

$$
\epsilon
$$

The RMS calculation becomes:

$$
\boxed{
RMS(x)
=
\sqrt{
\frac{1}{d}
\sum_{i=1}^{d}x_i^2
+
\epsilon
}
}
$$

### Why Do We Need Epsilon?

Consider:

$$
x=[0,0,0,0]
$$

Then:

$$
\frac{1}{d}
\sum x_i^2
=
0
$$

Without epsilon:

$$
\frac{x}{\sqrt{0}}
$$

would cause division by zero.

Therefore, we add a small value:

$$
\epsilon
$$

This ensures the denominator never becomes exactly zero.

---

### Gamma

RMSNorm also contains a learnable scaling parameter:

$$
\gamma
$$

After normalization:

$$
\hat{x}
=
\frac{x}{RMS(x)}
$$

we multiply element-wise with gamma:

$$
\boxed{
y
=
\gamma
\odot
\hat{x}
}
$$

where:

$$
\odot
$$

means element-wise multiplication.

For example, suppose:

$$
\hat{x}
=
[0.365,0.730,1.095,1.461]
$$

Initially:

$$
\gamma
=
[1,1,1,1]
$$

Then the output remains:

$$
[0.365,0.730,1.095,1.461]
$$

But during training, the model may learn:

$$
\gamma
=
[2,0.5,1.5,1]
$$

Then:

$$
[2,0.5,1.5,1]
\odot
[0.365,0.730,1.095,1.461]
$$

becomes:

$$
\boxed{
[0.730,0.365,1.643,1.461]
}
$$

Therefore, gamma allows the model to learn how strongly each dimension should be scaled.

---

### Complete RMSNorm Formula

Putting everything together:

$$
\boxed{
\text{RMSNorm}(x)
=
\gamma
\odot
\frac{x}
{
\sqrt{
\frac{1}{d}
\sum_{i=1}^{d}x_i^2
+
\epsilon
}
}
}
$$

where:

- $x$ = token representation
- $d$ = number of dimensions
- $\epsilon$ = small value for numerical stability
- $\gamma$ = learnable scaling parameter
- $\odot$ = element-wise multiplication

---

### Does RMSNorm Use Beta?

LayerNorm typically has:

$$
\gamma
$$

for scaling and:

$$
\beta
$$

for shifting.

Its output can be written as:

$$
\text{Output}
=
\gamma
\odot
\hat{x}
+
\beta
$$

RMSNorm typically uses only:

$$
\boxed{
\gamma
}
$$

and does not use beta.

Therefore:

$$
\boxed{
\text{RMSNorm}
=
\gamma
\odot
\frac{x}{RMS(x)}
}
$$

This makes RMSNorm simpler than LayerNorm.

---

## RMSNorm vs LayerNorm Summary

| Feature | LayerNorm | RMSNorm |
|---|---|---|
| Calculate mean | Yes | No |
| Subtract mean | Yes | No |
| Calculate variance | Yes | No |
| Calculate RMS | No | Yes |
| Centers around zero | Yes | No |
| Controls magnitude | Yes | Yes |
| Gamma | Yes | Yes |
| Beta | Usually yes | Usually no |
| Complexity | Higher | Lower |

---

## Final Mental Model

### LayerNorm

    Input
      ↓
    Calculate Mean
      ↓
    Subtract Mean
      ↓
    Normalize Spread
      ↓
    Gamma + Beta

Think:

> Center the representation and normalize it.

---

### RMSNorm

    Input
      ↓
    Calculate Overall Magnitude using RMS
      ↓
    Divide by RMS
      ↓
    Gamma

Think:

> Keep the representation direction but control its magnitude.

---

## Key Takeaway

The core RMSNorm operation is:

$$
\boxed{
\frac{x}{RMS(x)}
}
$$

where:

$$
\boxed{
RMS(x)
=
\sqrt{
\frac{1}{d}
\sum x_i^2
+
\epsilon
}
}
$$

The final output is:

$$
\boxed{
\text{RMSNorm}(x)
=
\gamma
\odot
\frac{x}{RMS(x)}
}
$$

The important property we will verify next is:

$$
\boxed{
RMS(\text{normalized output})
\approx1
}
$$

In [104]:
import torch

# Input representation
x = torch.tensor([2.0, 4.0, 6.0, 8.0])

# Learnable scale parameter (initially all 1)
gamma = torch.ones(4)

# Small value for numerical stability
eps = 1e-5


# Step 1: Square the values
squared = x ** 2

# Step 2: Calculate mean of squares
mean_squared = squared.mean()

# Step 3: Calculate RMS
rms = torch.sqrt(mean_squared + eps)

# Step 4: Normalize
normalized = x / rms

# Step 5: Apply gamma
output = normalized * gamma


print("Input:", x)
print()

print("Squared:", squared)
print("Mean of squares:", mean_squared)
print("RMS:", rms)
print()

print("Normalized:", normalized)
print("Gamma:", gamma)
print("Final output:", output)

Input: tensor([2., 4., 6., 8.])

Squared: tensor([ 4., 16., 36., 64.])
Mean of squares: tensor(30.)
RMS: tensor(5.4772)

Normalized: tensor([0.3651, 0.7303, 1.0954, 1.4606])
Gamma: tensor([1., 1., 1., 1.])
Final output: tensor([0.3651, 0.7303, 1.0954, 1.4606])


In [105]:
# Calculate RMS of normalized output

output_squared = normalized ** 2

output_mean_squared = output_squared.mean()

output_rms = torch.sqrt(output_mean_squared)

print("Normalized output:", normalized)
print("Squared output:", output_squared)
print("Mean of squared output:", output_mean_squared)
print("Output RMS:", output_rms)

Normalized output: tensor([0.3651, 0.7303, 1.0954, 1.4606])
Squared output: tensor([0.1333, 0.5333, 1.2000, 2.1333])
Mean of squared output: tensor(1.0000)
Output RMS: tensor(1.0000)


In [106]:
import torch

x = torch.tensor([2.0, 4.0, 6.0, 8.0])

# -------------------
# LayerNorm manually
# -------------------

mean = x.mean()
variance = ((x - mean) ** 2).mean()

layernorm = (x - mean) / torch.sqrt(variance)


# -------------------
# RMSNorm manually
# -------------------

rms = torch.sqrt((x ** 2).mean())

rmsnorm = x / rms


# -------------------
# Compare
# -------------------

print("Original:", x)
print()

print("LAYER NORM")
print("Output:", layernorm)
print("Mean:", layernorm.mean())
print("Variance:", (layernorm ** 2).mean())
print("RMS:", torch.sqrt((layernorm ** 2).mean()))

print()

print("RMS NORM")
print("Output:", rmsnorm)
print("Mean:", rmsnorm.mean())
print("Variance:", ((rmsnorm - rmsnorm.mean()) ** 2).mean())
print("RMS:", torch.sqrt((rmsnorm ** 2).mean()))

Original: tensor([2., 4., 6., 8.])

LAYER NORM
Output: tensor([-1.3416, -0.4472,  0.4472,  1.3416])
Mean: tensor(0.)
Variance: tensor(1.0000)
RMS: tensor(1.0000)

RMS NORM
Output: tensor([0.3651, 0.7303, 1.0954, 1.4606])
Mean: tensor(0.9129)
Variance: tensor(0.1667)
RMS: tensor(1.0000)


## 1.18.7 Implementing RMSNorm Manually

RMSNorm can be implemented step by step using:

$$
x=[2,4,6,8]
$$

Initially, let:

$$
\gamma=[1,1,1,1]
$$

and:

$$
\epsilon=10^{-5}
$$

### Step 1: Square the Input

$$
x^2
=
[4,16,36,64]
$$

### Step 2: Calculate Mean of Squares

$$
\frac{4+16+36+64}{4}
=
30
$$

### Step 3: Calculate RMS

$$
RMS(x)
=
\sqrt{30+\epsilon}
$$

$$
\approx5.4772
$$

### Step 4: Normalize

$$
\hat{x}
=
\frac{x}{RMS(x)}
$$

$$
=
\frac{[2,4,6,8]}{5.4772}
$$

Result:

$$
\boxed{
[0.3651,0.7303,1.0954,1.4606]
}
$$

### Step 5: Apply Gamma

$$
output
=
\gamma
\odot
\hat{x}
$$

Initially:

$$
\gamma=[1,1,1,1]
$$

Therefore:

$$
\boxed{
output=
[0.3651,0.7303,1.0954,1.4606]
}
$$

---

## 1.18.8 Why Does RMSNorm Produce RMS = 1?

The normalized vector is:

$$
\hat{x}
=
\frac{x}{RMS(x)}
$$

We know:

$$
RMS(x)
=
\sqrt{E(x^2)}
$$

Therefore:

$$
RMS(x)^2
=
E(x^2)
$$

Now calculate the RMS of the normalized vector:

$$
RMS(\hat{x})
=
\sqrt{
E(\hat{x}^2)
}
$$

Substitute:

$$
\hat{x}
=
\frac{x}{RMS(x)}
$$

Therefore:

$$
RMS(\hat{x})
=
\sqrt{
E
\left(
\frac{x^2}{RMS(x)^2}
\right)
}
$$

Since RMS is a constant for the vector:

$$
=
\sqrt{
\frac{E(x^2)}
{RMS(x)^2}
}
$$

But:

$$
RMS(x)^2
=
E(x^2)
$$

Therefore:

$$
=
\sqrt{
\frac{E(x^2)}
{E(x^2)}
}
$$

$$
=
\sqrt{1}
$$

Thus:

$$
\boxed{
RMS(\hat{x})=1
}
$$

This is not a coincidence.

We divide the vector by its RMS specifically so that the resulting vector has:

$$
\boxed{
RMS=1
}
$$

---

## 1.18.9 RMSNorm vs LayerNorm

Consider:

$$
x=[2,4,6,8]
$$

### LayerNorm Output

$$
\boxed{
[-1.3416,-0.4472,0.4472,1.3416]
}
$$

Properties:

$$
\boxed{
Mean=0
}
$$

$$
\boxed{
Variance=1
}
$$

$$
\boxed{
RMS=1
}
$$

LayerNorm performs:

```text
Input
  ↓
Subtract Mean
  ↓
Mean becomes 0
  ↓
Normalize Spread
  ↓
Variance becomes 1


## Next: Where RMSNorm Actually Sits in a Modern Transformer

Now we have learned RMSNorm itself. The next important question is:

> **Where exactly does RMSNorm go relative to Attention, FFN, and Residual connections?**

This is where we'll connect everything we've learned into the actual modern decoder architecture, including the difference between:

- **Post-Norm**, used in the original Transformer
- **Pre-Norm**, used by many modern LLMs
- How a SmolLM2-style block is structured

This is the right next step before we assemble our full Transformer block.

# 1.19 Where Does Normalization Sit in a Transformer?

We now understand the major components of a Transformer:

- Multi-Head Attention
- Residual Connections
- LayerNorm
- RMSNorm
- FFN
- SwiGLU

The next question is:

> In what order are these components connected inside a Transformer block?

There are two important approaches:

1. Post-Norm
2. Pre-Norm

---

## 1.19.1 Post-Norm: Original Transformer

The original Transformer architecture uses normalization after the main operation and residual addition.

The flow is:

    Input X
       ↓
    Multi-Head Attention
       ↓
    Residual Addition
       ↓
    LayerNorm
       ↓
    FFN
       ↓
    Residual Addition
       ↓
    LayerNorm

For the Attention part:

$$
\boxed{
Y
=
LayerNorm(
X
+
Attention(X)
)
}
$$

For the FFN part:

$$
\boxed{
Output
=
LayerNorm(
Y
+
FFN(Y)
)
}
$$

This is called Post-Norm because:

> Normalization happens after the transformation and residual addition.

The structure is:

    Transformation
         ↓
    Residual Addition
         ↓
    Normalization

---

## 1.19.2 Pre-Norm: Modern LLM Approach

Many modern LLM architectures use Pre-Norm.

Here, normalization happens before Attention or FFN.

### Attention Block

The flow is:

                 Input X
                    │
             ┌──────┴──────┐
             │             │
             ↓             │
          RMSNorm          │
             ↓             │
         Attention         │
             ↓             │
             └────── Add ◄─┘
                      │
                      ↓
                      Y

The formula is:

$$
\boxed{
Y
=
X
+
Attention(
RMSNorm(X)
)
}
$$

The important point is:

The original input:

$$
X
$$

flows directly through the residual connection.

Only the Attention branch receives the normalized input.

---

### FFN Block

The output from the Attention block is:

$$
Y
$$

Then:

                  Y
                  │
           ┌──────┴──────┐
           │             │
           ↓             │
        RMSNorm          │
           ↓             │
        SwiGLU FFN       │
           ↓             │
           └────── Add ◄─┘
                    │
                    ↓
                  Output

The formula is:

$$
\boxed{
Output
=
Y
+
FFN(
RMSNorm(Y)
)
}
$$

Again:

- One path goes directly through the residual connection.
- One path goes through RMSNorm and FFN.
- Both paths are added together.

---

## 1.19.3 Post-Norm vs Pre-Norm

### Post-Norm

$$
\boxed{
Y
=
LayerNorm(
X
+
Attention(X)
)
}
$$

Flow:

    X
    ↓
    Attention
    ↓
    Add Original X
    ↓
    Normalize

Think:

> Transform → Add → Normalize

---

### Pre-Norm

$$
\boxed{
Y
=
X
+
Attention(
RMSNorm(X)
)
}
$$

Flow:

    X
    ↓
    Normalize
    ↓
    Transform
    ↓
    Add Original X

Think:

> Normalize → Transform → Add Original

---

## 1.19.4 The Residual Path in Pre-Norm

Consider:

$$
Y
=
X
+
Attention(
RMSNorm(X)
)
$$

The input X follows two paths.

### Path 1: Residual Path

$$
X
$$

flows directly to the final addition.

### Path 2: Transformation Path

$$
X
\rightarrow
RMSNorm
\rightarrow
Attention
$$

The two paths are combined:

$$
\boxed{
Y
=
Residual
+
Transformation
}
$$

Visually:

                 X
               /   \
              /     \
             ↓       ───────────────┐
          RMSNorm                   │
             ↓                      │
         Attention                  │
             ↓                      │
             └────────── Add ◄──────┘
                          │
                          ↓
                          Y

This means the original representation has a direct path through the Transformer block.

---

## 1.19.5 Complete Modern Decoder Block

A simplified modern decoder block looks like:

                     Input X
                        │
             ┌──────────┴──────────┐
             │                     │
             ↓                  Residual
          RMSNorm                  │
             ↓                     │
    Causal Multi-Head Attention     │
             ↓                     │
             └────────── Add ◄─────┘
                          │
                          ↓
                          Y
                          │
             ┌────────────┴──────────┐
             │                       │
             ↓                    Residual
          RMSNorm                    │
             ↓                       │
          SwiGLU FFN                 │
             ↓                       │
             └─────────── Add ◄──────┘
                           │
                           ↓
                         Output

The equations are:

### Attention Block

$$
\boxed{
Y
=
X
+
Attention(
RMSNorm(X)
)
}
$$

### FFN Block

$$
\boxed{
Output
=
Y
+
FFN(
RMSNorm(Y)
)
}
$$

---

## Key Takeaway

### Post-Norm

$$
\boxed{
Transform
\rightarrow
Add
\rightarrow
Normalize
}
$$

### Pre-Norm

$$
\boxed{
Normalize
\rightarrow
Transform
\rightarrow
Add
}
$$

Modern LLM architectures commonly use Pre-Norm because the original representation has a cleaner residual path through the network.

The key Pre-Norm equation is:

$$
\boxed{
Y
=
X
+
Attention(
RMSNorm(X)
)
}
$$

This equation connects three concepts we already learned:

1. RMSNorm controls the scale of the representation.
2. Attention transforms the normalized representation.
3. The residual connection adds the transformed information back to the original representation.

In [107]:
import torch

# --------------------------------
# Input representation
# --------------------------------

X = torch.tensor([2.0, 4.0, 6.0, 8.0])

# RMSNorm gamma
gamma = torch.ones(4)

eps = 1e-5


# --------------------------------
# Step 1: RMSNorm(X)
# --------------------------------

rms = torch.sqrt((X ** 2).mean() + eps)

X_norm = gamma * (X / rms)


# --------------------------------
# Step 2: Attention output
# --------------------------------

# Assume our Attention layer produced this
attention_output = torch.tensor([
    0.5,
   -0.2,
    0.3,
    0.1
])


# --------------------------------
# Step 3: Residual Addition
# --------------------------------

Y = X + attention_output


# --------------------------------
# Print everything
# --------------------------------

print("Original X:", X)
print()

print("RMS of X:", rms)
print("Normalized X:", X_norm)
print()

print("Attention output:", attention_output)
print()

print("Residual path:", X)
print("Attention path:", attention_output)
print()

print("Y = X + Attention(RMSNorm(X))")
print("Y:", Y)

Original X: tensor([2., 4., 6., 8.])

RMS of X: tensor(5.4772)
Normalized X: tensor([0.3651, 0.7303, 1.0954, 1.4606])

Attention output: tensor([ 0.5000, -0.2000,  0.3000,  0.1000])

Residual path: tensor([2., 4., 6., 8.])
Attention path: tensor([ 0.5000, -0.2000,  0.3000,  0.1000])

Y = X + Attention(RMSNorm(X))
Y: tensor([2.5000, 3.8000, 6.3000, 8.1000])


In [108]:
# --------------------------------
# FFN Block
# --------------------------------

# Step 1: RMSNorm(Y)

rms_Y = torch.sqrt((Y ** 2).mean() + eps)

Y_norm = gamma * (Y / rms_Y)


# --------------------------------
# Step 2: FFN output
# --------------------------------

# Assume SwiGLU FFN produced this
ffn_output = torch.tensor([
    0.2,
   -0.4,
    0.1,
    0.3
])


# --------------------------------
# Step 3: Residual Addition
# --------------------------------

output = Y + ffn_output


# --------------------------------
# Print
# --------------------------------

print()
print("---------- FFN BLOCK ----------")
print()

print("Input Y:", Y)
print("RMS of Y:", rms_Y)
print("Normalized Y:", Y_norm)
print()

print("FFN output:", ffn_output)
print()

print("Residual path:", Y)
print("FFN path:", ffn_output)
print()

print("Final Output:")
print(output)


---------- FFN BLOCK ----------

Input Y: tensor([2.5000, 3.8000, 6.3000, 8.1000])
RMS of Y: tensor(5.6123)
Normalized Y: tensor([0.4455, 0.6771, 1.1225, 1.4433])

FFN output: tensor([ 0.2000, -0.4000,  0.1000,  0.3000])

Residual path: tensor([2.5000, 3.8000, 6.3000, 8.1000])
FFN path: tensor([ 0.2000, -0.4000,  0.1000,  0.3000])

Final Output:
tensor([2.7000, 3.4000, 6.4000, 8.4000])


# 1.19.6 Complete Pre-Norm Transformer Block Flow

A modern Transformer decoder block contains two major sub-layers:

1. Attention
2. FFN

Each sub-layer uses:

- RMSNorm
- A main transformation
- A residual connection

---

## 1.19.6.1 Attention Block

The input representation is:

$$
X
$$

First, normalize `X`:

$$
X_{norm} = RMSNorm(X)
$$

Then send the normalized representation into Attention:

$$
Attention(X_{norm})
$$

Finally, add the Attention output back to the original `X`:

$$
\boxed{
Y = X + Attention(RMSNorm(X))
}
$$

### Flow

```text
                 X
                 │
          ┌──────┴──────┐
          │             │
          ↓             │
       RMSNorm          │
          ↓             │
       Attention        │
          ↓             │
          └────── + ◄───┘
                 │
                 ↓
                 Y
```

The important point:

> `RMSNorm(X)` is used only for the Attention path.

The original `X` travels directly through the residual path.

There are two paths:

### Transformation Path

```text
X
↓
RMSNorm
↓
Attention
```

### Residual Path

```text
X
↓
Directly to Addition
```

Then:

$$
\boxed{
Y = X + Attention(RMSNorm(X))
}
$$

---

## 1.19.6.2 FFN Block

Now the output from the Attention block, `Y`, enters the second sub-layer.

First:

$$
Y_{norm} = RMSNorm(Y)
$$

Then:

$$
FFN(Y_{norm})
$$

Finally, add the FFN output back to the original `Y`:

$$
\boxed{
Output = Y + FFN(RMSNorm(Y))
}
$$

### Flow

```text
                 Y
                 │
          ┌──────┴──────┐
          │             │
          ↓             │
       RMSNorm          │
          ↓             │
       SwiGLU FFN       │
          ↓             │
          └────── + ◄───┘
                 │
                 ↓
               Output
```

Again, there are two paths.

### Transformation Path

```text
Y
↓
RMSNorm
↓
SwiGLU FFN
```

### Residual Path

```text
Y
↓
Directly to Addition
```

Then:

$$
\boxed{
Output = Y + FFN(RMSNorm(Y))
}
$$

---

## 1.19.6.3 Complete Transformer Block

Combining both sub-layers:

```text
                     Input X
                        │
             ┌──────────┴──────────┐
             │                     │
             ↓                  Residual
          RMSNorm                  │
             ↓                     │
           Attention                │
             ↓                     │
             └────────── Add ◄─────┘
                          │
                          ↓
                          Y
                          │
             ┌────────────┴──────────┐
             │                       │
             ↓                    Residual
          RMSNorm                    │
             ↓                       │
          SwiGLU FFN                 │
             ↓                       │
             └─────────── Add ◄──────┘
                           │
                           ↓
                         Output
```

The complete conceptual flow is:

```text
Input
  ↓
RMSNorm
  ↓
Attention
  ↓
Residual Addition
  ↓
RMSNorm
  ↓
SwiGLU FFN
  ↓
Residual Addition
  ↓
Output
```

Remember that the simplified flow above does not visually show the residual path traveling alongside each transformation.

---

## 1.19.6.4 Complete Mathematical Form

### Attention Sub-Layer

The complete equation is:

$$
\boxed{
Y = X + Attention(RMSNorm(X))
}
$$

Breaking it down:

#### Step 1: Normalize

$$
X_{norm} = RMSNorm(X)
$$

#### Step 2: Apply Attention

$$
A = Attention(X_{norm})
$$

#### Step 3: Add the Original Input

$$
Y = X + A
$$

Therefore:

$$
\boxed{
Y = X + Attention(RMSNorm(X))
}
$$

---

### FFN Sub-Layer

The complete equation is:

$$
\boxed{
Output = Y + FFN(RMSNorm(Y))
}
$$

Breaking it down:

#### Step 1: Normalize

$$
Y_{norm} = RMSNorm(Y)
$$

#### Step 2: Apply FFN

$$
F = FFN(Y_{norm})
$$

#### Step 3: Add the Original Y

$$
Output = Y + F
$$

Therefore:

$$
\boxed{
Output = Y + FFN(RMSNorm(Y))
}
$$

---

## 1.19.6.5 SwiGLU FFN Mathematical Form

A SwiGLU FFN has three main projections:

1. Up projection
2. Gate projection
3. Down projection

The input is:

$$
x
$$

### Step 1: Information / Up Path

$$
Up = xW_{up}
$$

This creates the expanded information representation.

---

### Step 2: Gate Path

First:

$$
Gate_{raw} = xW_{gate}
$$

Apply SiLU:

$$
Gate = SiLU(Gate_{raw})
$$

Therefore:

$$
\boxed{
Gate = SiLU(xW_{gate})
}
$$

---

### Step 3: Gate the Information

Multiply element-wise:

$$
Combined = Up \odot Gate
$$

Therefore:

$$
\boxed{
Combined =
(xW_{up})
\odot
SiLU(xW_{gate})
}
$$

---

### Step 4: Down Projection

Project back to the original hidden dimension:

$$
FFN(x) = CombinedW_{down}
$$

Therefore:

$$
\boxed{
FFN(x) =
\left(
(xW_{up})
\odot
SiLU(xW_{gate})
\right)
W_{down}
}
$$

---

## 1.19.6.6 Complete Modern Transformer Block Formula

The Attention block is:

$$
\boxed{
Y = X + Attention(RMSNorm(X))
}
$$

The FFN block is:

$$
\boxed{
Output = Y + FFN(RMSNorm(Y))
}
$$

For a SwiGLU FFN:

$$
FFN(x) =
\left(
(xW_{up})
\odot
SiLU(xW_{gate})
\right)
W_{down}
$$

Therefore:

$$
\boxed{
Output =
Y +
\left[
(RMSNorm(Y)W_{up})
\odot
SiLU(RMSNorm(Y)W_{gate})
\right]
W_{down}
}
$$

where:

$$
\boxed{
Y = X + Attention(RMSNorm(X))
}
$$

This represents the complete high-level mathematical flow of our modern Pre-Norm Transformer block.

---

## 1.19.6.7 What Attention and FFN Each Do

The two major sub-layers have different jobs.

### Attention

Attention allows tokens to interact with other tokens.

Conceptually:

```text
Token 1 ──────┐
Token 2 ──────┼──► Attention
Token 3 ──────┤
Token 4 ──────┘
```

Attention combines information across tokens.

Its conceptual job is:

> Let tokens communicate with each other.

---

### FFN

After Attention, each token goes through the FFN independently.

Conceptually:

```text
Token 1 ───► FFN ───► Token 1 output

Token 2 ───► FFN ───► Token 2 output

Token 3 ───► FFN ───► Token 3 output

Token 4 ───► FFN ───► Token 4 output
```

The FFN does not directly mix tokens together.

Instead, it processes the features inside each individual token representation.

Its conceptual job is:

> Transform and process each token's internal representation.

---

## 1.19.6.8 Attention and FFN Together

This is an important mental model for understanding Transformers:

```text
TOKENS
  │
  ↓

ATTENTION

Tokens exchange information
with other tokens

  │
  ↓

FFN

Each token independently
processes its new information

  │
  ↓

OUTPUT
```

Therefore:

### Attention

```text
Across tokens
```

### FFN

```text
Within each token
```

Or:

```text
Attention
=
Token-to-token information exchange


FFN
=
Per-token feature processing
```

---

## 1.19.6.9 Why Residual Connections Are Important Here

Each sub-layer modifies the representation.

Without a residual connection:

```text
X
↓
Attention
↓
Y
```

The old representation is completely replaced.

With a residual connection:

```text
X ──────────────────┐
                    │
RMSNorm             │
↓                   │
Attention           │
↓                   │
──────────────► Add ◄
                    │
                    ▼
                    Y
```

Therefore:

$$
Y = X + Attention(RMSNorm(X))
$$

The model keeps the original information and adds new information.

The same happens in the FFN block:

$$
Output = Y + FFN(RMSNorm(Y))
$$

Each Transformer sub-layer can therefore be viewed as:

$$
\boxed{
NewRepresentation
=
OldRepresentation
+
NewLearnedInformation
}
$$

---

## 1.19.6.10 Final Mental Model

A modern Pre-Norm Transformer block works in two stages.

### Stage 1: Attention

```text
Normalize representation
        ↓
Tokens communicate
        ↓
Add original representation
```

Formula:

$$
\boxed{
Y = X + Attention(RMSNorm(X))
}
$$

---

### Stage 2: FFN

```text
Normalize representation
        ↓
Process token features
        ↓
Add previous representation
```

Formula:

$$
\boxed{
Output = Y + FFN(RMSNorm(Y))
}
$$

---

# Final Complete Picture

```text
                        INPUT X
                           │
              ┌────────────┴────────────┐
              │                         │
              │                    Residual
              ↓                         │
           RMSNorm                      │
              ↓                         │
           Attention                    │
              ↓                         │
              └─────────── Add ◄────────┘
                            │
                            ▼
                            Y
                            │
              ┌─────────────┴─────────────┐
              │                           │
              │                      Residual
              ↓                           │
           RMSNorm                        │
              ↓                           │
          SwiGLU FFN                      │
              ↓                           │
              └──────────── Add ◄─────────┘
                             │
                             ▼
                           OUTPUT
```

---

# Key Equations to Remember

## Attention Block

$$
\boxed{
Y = X + Attention(RMSNorm(X))
}
$$

## SwiGLU FFN

$$
\boxed{
FFN(x) =
\left(
(xW_{up})
\odot
SiLU(xW_{gate})
\right)
W_{down}
}
$$

## FFN Block

$$
\boxed{
Output = Y + FFN(RMSNorm(Y))
}
$$

---

# Final Summary

| Component | Main Job |
|---|---|
| RMSNorm | Controls representation scale |
| Attention | Allows tokens to exchange information |
| SwiGLU FFN | Processes features within each token |
| Residual Connection | Preserves previous information and adds new information |

The complete idea is:

```text
Normalize
    ↓
Tokens communicate
    ↓
Preserve + Add

    ↓

Normalize
    ↓
Each token processes features
    ↓
Preserve + Add
```

This completes the conceptual structure of a modern Pre-Norm Transformer decoder block.

In [109]:
import torch

X = torch.tensor([
    [1.0, 2.0, 3.0, 4.0],
    [2.0, 4.0, 6.0, 8.0],
    [1.0, 3.0, 5.0, 7.0],
    [4.0, 3.0, 2.0, 1.0]
])

print("Input X:")
print(X)

print("\nShape:", X.shape)

Input X:
tensor([[1., 2., 3., 4.],
        [2., 4., 6., 8.],
        [1., 3., 5., 7.],
        [4., 3., 2., 1.]])

Shape: torch.Size([4, 4])


In [112]:
import torch

X = torch.tensor([
    [1.0, 2.0, 3.0, 4.0],
    [2.0, 4.0, 6.0, 8.0],
    [1.0, 3.0, 5.0, 7.0],
    [4.0, 3.0, 2.0, 1.0]
])

eps = 1e-5

# Square each value
squared = X ** 2

# Calculate mean of squares for each token
mean_squared = squared.mean(dim=-1, keepdim=True)

# Calculate RMS for each token
rms = torch.sqrt(mean_squared + eps)

# Normalize each token
X_norm = X / rms

print("Original X:")
print(X)

print("\nSquared:")
print(squared)

print("\nMean of squares per token:")
print(mean_squared)

print("\nRMS per token:")
print(rms)

print("\nNormalized X:")
print(X_norm)

print("\nShape of normalized X:")
print(X_norm.shape)

Original X:
tensor([[1., 2., 3., 4.],
        [2., 4., 6., 8.],
        [1., 3., 5., 7.],
        [4., 3., 2., 1.]])

Squared:
tensor([[ 1.,  4.,  9., 16.],
        [ 4., 16., 36., 64.],
        [ 1.,  9., 25., 49.],
        [16.,  9.,  4.,  1.]])

Mean of squares per token:
tensor([[ 7.5000],
        [30.0000],
        [21.0000],
        [ 7.5000]])

RMS per token:
tensor([[2.7386],
        [5.4772],
        [4.5826],
        [2.7386]])

Normalized X:
tensor([[0.3651, 0.7303, 1.0954, 1.4606],
        [0.3651, 0.7303, 1.0954, 1.4606],
        [0.2182, 0.6547, 1.0911, 1.5275],
        [1.4606, 1.0954, 0.7303, 0.3651]])

Shape of normalized X:
torch.Size([4, 4])


In [113]:
# Q projection matrix
W_Q = torch.tensor([
    [1.0, 0.0, 0.5, 0.0],
    [0.0, 1.0, 0.0, 0.5],
    [0.5, 0.0, 1.0, 0.0],
    [0.0, 0.5, 0.0, 1.0]
])

# K projection matrix
W_K = torch.tensor([
    [1.0, 0.5, 0.0, 0.0],
    [0.0, 1.0, 0.5, 0.0],
    [0.0, 0.0, 1.0, 0.5],
    [0.5, 0.0, 0.0, 1.0]
])

# V projection matrix
W_V = torch.tensor([
    [1.0, 0.0, 0.0, 0.5],
    [0.0, 1.0, 0.5, 0.0],
    [0.5, 0.0, 1.0, 0.0],
    [0.0, 0.5, 0.0, 1.0]
])

Q = X_norm @ W_Q
K = X_norm @ W_K
V = X_norm @ W_V

print("X_norm:")
print(X_norm)

print("\nQ:")
print(Q)
print("Q Shape:", Q.shape)

print("\nK:")
print(K)
print("K Shape:", K.shape)

print("\nV:")
print(V)
print("V Shape:", V.shape)

X_norm:
tensor([[0.3651, 0.7303, 1.0954, 1.4606],
        [0.3651, 0.7303, 1.0954, 1.4606],
        [0.2182, 0.6547, 1.0911, 1.5275],
        [1.4606, 1.0954, 0.7303, 0.3651]])

Q:
tensor([[0.9129, 1.4606, 1.2780, 1.8257],
        [0.9129, 1.4606, 1.2780, 1.8257],
        [0.7638, 1.4184, 1.2002, 1.8549],
        [1.8257, 1.2780, 1.4606, 0.9129]])
Q Shape: torch.Size([4, 4])

K:
tensor([[1.0954, 0.9129, 1.4606, 2.0083],
        [1.0954, 0.9129, 1.4606, 2.0083],
        [0.9820, 0.7638, 1.4184, 2.0731],
        [1.6432, 1.8257, 1.2780, 0.7303]])
K Shape: torch.Size([4, 4])

V:
tensor([[0.9129, 1.4606, 1.4606, 1.6432],
        [0.9129, 1.4606, 1.4606, 1.6432],
        [0.7638, 1.4184, 1.4184, 1.6366],
        [1.8257, 1.2780, 1.2780, 1.0954]])
V Shape: torch.Size([4, 4])


In [114]:
num_heads = 2
head_dim = 2

# Q: [4, 4]
# Reshape into [tokens, heads, head_dim]
Q_heads = Q.view(4, num_heads, head_dim)

# Move heads to the first dimension
# [tokens, heads, head_dim]
#        ↓
# [heads, tokens, head_dim]

Q_heads = Q_heads.transpose(0, 1)


K_heads = K.view(4, num_heads, head_dim)
K_heads = K_heads.transpose(0, 1)


V_heads = V.view(4, num_heads, head_dim)
V_heads = V_heads.transpose(0, 1)


print("Q Heads:")
print(Q_heads)
print("Q Heads Shape:", Q_heads.shape)

print("\nK Heads:")
print(K_heads)
print("K Heads Shape:", K_heads.shape)

print("\nV Heads:")
print(V_heads)
print("V Heads Shape:", V_heads.shape)

Q Heads:
tensor([[[0.9129, 1.4606],
         [0.9129, 1.4606],
         [0.7638, 1.4184],
         [1.8257, 1.2780]],

        [[1.2780, 1.8257],
         [1.2780, 1.8257],
         [1.2002, 1.8549],
         [1.4606, 0.9129]]])
Q Heads Shape: torch.Size([2, 4, 2])

K Heads:
tensor([[[1.0954, 0.9129],
         [1.0954, 0.9129],
         [0.9820, 0.7638],
         [1.6432, 1.8257]],

        [[1.4606, 2.0083],
         [1.4606, 2.0083],
         [1.4184, 2.0731],
         [1.2780, 0.7303]]])
K Heads Shape: torch.Size([2, 4, 2])

V Heads:
tensor([[[0.9129, 1.4606],
         [0.9129, 1.4606],
         [0.7638, 1.4184],
         [1.8257, 1.2780]],

        [[1.4606, 1.6432],
         [1.4606, 1.6432],
         [1.4184, 1.6366],
         [1.2780, 1.0954]]])
V Heads Shape: torch.Size([2, 4, 2])


In [115]:
import math

scores = Q_heads @ K_heads.transpose(-2, -1)
scaled_scores = scores / math.sqrt(head_dim)

print("Scores:")
print(scores)
print("Shape:", scores.shape)

print("\nScaled scores:")
print(scaled_scores)

Scores:
tensor([[[2.3333, 2.3333, 2.0120, 4.1667],
         [2.3333, 2.3333, 2.0120, 4.1667],
         [2.1315, 2.1315, 1.8333, 3.8446],
         [3.1667, 3.1667, 2.7689, 5.3333]],

        [[5.5333, 5.5333, 5.5976, 2.9667],
         [5.5333, 5.5333, 5.5977, 2.9667],
         [5.4781, 5.4781, 5.5476, 2.8885],
         [3.9667, 3.9667, 3.9642, 2.5333]]])
Shape: torch.Size([2, 4, 4])

Scaled scores:
tensor([[[1.6499, 1.6499, 1.4227, 2.9463],
         [1.6499, 1.6499, 1.4227, 2.9463],
         [1.5072, 1.5072, 1.2964, 2.7186],
         [2.2392, 2.2392, 1.9579, 3.7712]],

        [[3.9127, 3.9127, 3.9581, 2.0977],
         [3.9127, 3.9127, 3.9581, 2.0977],
         [3.8736, 3.8736, 3.9228, 2.0425],
         [2.8049, 2.8049, 2.8031, 1.7913]]])


# 1.20.3.4 Causal Masking

After calculating the scaled attention scores:

$$
\text{Scaled Scores}
=
\frac{QK^T}{\sqrt{d_k}}
$$

we need to prevent a decoder-only language model from looking at future tokens.

## Example

For four tokens:

```text
T1 → The
T2 → cat
T3 → sat
T4 → down
```

The allowed attention pattern is:

```text
             Keys

             T1   T2   T3   T4

Query T1      ✓    ✗    ✗    ✗

Query T2      ✓    ✓    ✗    ✗

Query T3      ✓    ✓    ✓    ✗

Query T4      ✓    ✓    ✓    ✓
```

This is called **causal attention**.

## Why Mask Future Tokens?

When predicting the next token, the model should only use:

- Previous tokens
- The current token

It must not see future tokens.

## The Causal Mask

We create:

```text
[
 [False, True,  True,  True ],
 [False, False, True,  True ],
 [False, False, False, True ],
 [False, False, False, False]
]
```

Where:

```text
False → Allowed
True  → Mask / Block
```

Masked positions are replaced with:

$$
-\infty
$$

Example:

```text
Before:

[
 [1.65, 1.65, 1.42, 2.95],
 [1.65, 1.65, 1.42, 2.95],
 [1.51, 1.51, 1.30, 2.72],
 [2.24, 2.24, 1.96, 3.77]
]
```

After applying the causal mask:

```text
[
 [1.65, -∞,   -∞,   -∞ ],
 [1.65, 1.65, -∞,   -∞ ],
 [1.51, 1.51, 1.30, -∞ ],
 [2.24, 2.24, 1.96, 3.77]
]
```

## Why Use $-\infty$?

After masking, we apply Softmax:

$$
\text{Softmax}(x_i)
=
\frac{e^{x_i}}
{\sum_j e^{x_j}}
$$

For a masked position:

$$
e^{-\infty}=0
$$

Therefore:

```text
Masked position
      ↓
     -∞
      ↓
   Softmax
      ↓
      0
```

The model gives zero attention probability to future tokens.

## PyTorch Implementation

```python
mask = torch.triu(
    torch.ones(4, 4, dtype=torch.bool),
    diagonal=1
)

masked_scores = scaled_scores.masked_fill(
    mask,
    float("-inf")
)
```

The same $[4 \times 4]$ mask is automatically applied to both attention heads.

## Complete Flow

$$
Q,K
\rightarrow
QK^T
\rightarrow
\frac{QK^T}{\sqrt{d_k}}
\rightarrow
\text{Causal Mask}
\rightarrow
\text{Softmax}
$$

In [116]:
mask = torch.triu(
    torch.ones(4, 4, dtype=torch.bool),
    diagonal=1
)

masked_scores = scaled_scores.masked_fill(mask, float("-inf"))

print("Mask:")
print(mask)

print("\nMasked scores:")
print(masked_scores)

Mask:
tensor([[False,  True,  True,  True],
        [False, False,  True,  True],
        [False, False, False,  True],
        [False, False, False, False]])

Masked scores:
tensor([[[1.6499,   -inf,   -inf,   -inf],
         [1.6499, 1.6499,   -inf,   -inf],
         [1.5072, 1.5072, 1.2964,   -inf],
         [2.2392, 2.2392, 1.9579, 3.7712]],

        [[3.9127,   -inf,   -inf,   -inf],
         [3.9127, 3.9127,   -inf,   -inf],
         [3.8736, 3.8736, 3.9228,   -inf],
         [2.8049, 2.8049, 2.8031, 1.7913]]])


# 1.20.3.5 Softmax: Converting Scores into Attention Probabilities

After applying the causal mask, we have masked attention scores.

These scores are still raw compatibility values. We need to convert them into attention probabilities.

## Softmax Formula

$$
\text{Softmax}(x_i)
=
\frac{e^{x_i}}
{\sum_j e^{x_j}}
$$

Softmax converts the attention scores into probabilities.

Each attention row will have:

- Positive values
- Values between 0 and 1
- Sum of all values equal to 1

## Example

Before Softmax:

```text
[1.5072, 1.5072, 1.2964, -inf]
```

After Softmax:

```text
[0.356, 0.356, 0.289, 0.000]
```

The masked position receives:

$$
e^{-\infty}=0
$$

Therefore its attention probability becomes:

$$
0
$$

## What Does Softmax Mean?

Before Softmax:

```text
How compatible is each Key with this Query?
```

After Softmax:

```text
How should this Query distribute its attention?
```

For example:

```text
Token 3:

35.6% → Token 1
35.6% → Token 2
28.9% → Token 3
0%    → Token 4
```

## PyTorch

```python
attention_weights = F.softmax(masked_scores, dim=-1)

print(attention_weights)
print(attention_weights.sum(dim=-1))
```

## Why `dim=-1`?

The attention tensor has shape:

$$
[Heads,\ Query,\ Key]
$$

For our example:

$$
[2,\ 4,\ 4]
$$

Softmax is applied across the last dimension, the Key dimension.

Each Query therefore distributes its attention across all allowed Keys.

## Attention Flow

$$
QK^T
\rightarrow
\frac{QK^T}{\sqrt{d_k}}
\rightarrow
\text{Causal Mask}
\rightarrow
\text{Softmax}
\rightarrow
\text{Attention Weights}
$$

In [120]:
attention_weights = F.softmax(masked_scores, dim=-1)

print("Attention weights:")
print(attention_weights)

print("\nSum across keys:")
print(attention_weights.sum(dim=-1))

Attention weights:
tensor([[[1.0000, 0.0000, 0.0000, 0.0000],
         [0.5000, 0.5000, 0.0000, 0.0000],
         [0.3559, 0.3559, 0.2882, 0.0000],
         [0.1355, 0.1355, 0.1022, 0.6268]],

        [[1.0000, 0.0000, 0.0000, 0.0000],
         [0.5000, 0.5000, 0.0000, 0.0000],
         [0.3278, 0.3278, 0.3443, 0.0000],
         [0.2975, 0.2975, 0.2970, 0.1080]]])

Sum across keys:
tensor([[1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000]])


# 1.20.3.6 Attention Output: Multiplying Attention Weights with V

After Softmax, we have attention weights.

These weights tell us:

```text
Where should each token look?
```

But we still need to collect the actual information.

That information comes from the Value vectors.

## Q, K and V Intuition

```text
Q → What am I looking for?

K → How relevant is this token?

Q × K → How relevant are tokens to each other?

Softmax → How much attention should each token receive?

V → What information should be collected?
```

## Attention Output Formula

$$
\text{Attention Output}
=
\text{Attention Weights} \times V
$$

Or the complete attention formula:

$$
\text{Attention}(Q,K,V)
=
\text{Softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
+
\text{Mask}
\right)
V
$$

## Shapes Per Head

Attention weights:

$$
[4 \times 4]
$$

Value vectors:

$$
[4 \times 2]
$$

Matrix multiplication:

$$
[4 \times 4]
\times
[4 \times 2]
=
[4 \times 2]
$$

## Overall Shapes

For two heads:

$$
\text{Attention Weights}
=
[2,\ 4,\ 4]
$$

$$
V
=
[2,\ 4,\ 2]
$$

Therefore:

$$
\text{Attention Output}
=
[2,\ 4,\ 2]
$$

## Intuition

The attention weights determine how much information to take from each Value vector.

For example:

```text
Attention weights:

[0.2, 0.3, 0.5]

Values:

V1
V2
V3
```

The output becomes:

$$
0.2V_1
+
0.3V_2
+
0.5V_3
$$

Therefore, attention creates a weighted combination of information from different tokens.

## PyTorch

```python
attention_output = attention_weights @ V_heads

print(attention_output)
print(attention_output.shape)
```

## Complete Attention Flow

$$
X
\rightarrow
Q,K,V
\rightarrow
\text{Split into Heads}
\rightarrow
QK^T
\rightarrow
\text{Scale}
\rightarrow
\text{Causal Mask}
\rightarrow
\text{Softmax}
\rightarrow
\text{Attention Weights}
\rightarrow
AttentionWeights \times V
\rightarrow
\text{Attention Output}
$$

In [121]:
attention_output = attention_weights @ V_heads

print("Attention output:")
print(attention_output)

print("\nShape:")
print(attention_output.shape)

Attention output:
tensor([[[0.9129, 1.4606],
         [0.9129, 1.4606],
         [0.8699, 1.4484],
         [1.4699, 1.3418]],

        [[1.4606, 1.6432],
         [1.4606, 1.6432],
         [1.4461, 1.6409],
         [1.4284, 1.5821]]])

Shape:
torch.Size([2, 4, 2])


# 1.20.3.7 Combining Multiple Attention Heads

After calculating attention for each head, we have:

$$
\text{Attention Output Shape}
=
[2,\ 4,\ 2]
$$

This means:

- 2 attention heads
- 4 tokens
- 2 dimensions per head

## Head Outputs

Each token currently has a separate representation from each head.

For example:

```text
Token 1:

Head 1 → [0.9129, 1.4606]

Head 2 → [1.4606, 1.6432]
```

We combine them by concatenating the head dimensions:

```text
[0.9129, 1.4606]

+

[1.4606, 1.6432]

↓

[0.9129, 1.4606, 1.4606, 1.6432]
```

## Dimension

We have:

$$
\text{Number of Heads}
=
2
$$

and:

$$
d_{head}
=
2
$$

Therefore:

$$
d_{model}
=
2 \times 2
=
4
$$

After combining heads:

$$
[2,\ 4,\ 2]
\rightarrow
[4,\ 4]
$$

The final shape is:

$$
[\text{Tokens},\ d_{model}]
$$

## PyTorch

```python
combined_heads = attention_output.transpose(0, 1).reshape(4, 4)

print(combined_heads)
print(combined_heads.shape)
```

## Why Transpose?

Before:

$$
[\text{Head},\ \text{Token},\ d_{head}]
$$

We change it to:

$$
[\text{Token},\ \text{Head},\ d_{head}]
$$

Then concatenate the head dimensions.

## Flow

$$
[Heads,\ Tokens,\ d_{head}]
\rightarrow
[Tokens,\ Heads,\ d_{head}]
\rightarrow
[Tokens,\ Heads \times d_{head}]
$$

Therefore:

$$
[2,\ 4,\ 2]
\rightarrow
[4,\ 2,\ 2]
\rightarrow
[4,\ 4]
$$

In [122]:
combined_heads = attention_output.transpose(0, 1).reshape(4, 4)

print("Combined heads:")
print(combined_heads)

print("\nShape:")
print(combined_heads.shape)

Combined heads:
tensor([[0.9129, 1.4606, 1.4606, 1.6432],
        [0.9129, 1.4606, 1.4606, 1.6432],
        [0.8699, 1.4484, 1.4461, 1.6409],
        [1.4699, 1.3418, 1.4284, 1.5821]])

Shape:
torch.Size([4, 4])


# 1.20.3.8 Output Projection $W_O$

After calculating attention separately for each head, we combine the heads.

The combined output has shape:

$$
[\text{Tokens}, d_{model}]
$$

In our example:

$$
[4,4]
$$

However, Multi-Head Attention has one final learned projection.

## Output Projection

The formula is:

$$
\text{MHA Output}
=
\text{Combined Heads} \times W_O
$$

where:

$$
W_O
=
[d_{model},d_{model}]
$$

In our example:

$$
W_O
=
[4,4]
$$

Therefore:

$$
[4,4]
\times
[4,4]
=
[4,4]
$$

## Why Do We Need $W_O$?

After concatenating heads:

```text
[Head 1 information | Head 2 information]
```

For example:

```text
[a, b | c, d]
```

The information from each head is simply placed next to each other.

The output projection allows the model to learn how to combine information from different heads.

```text
Head 1 → Feature A

Head 2 → Feature B

        ↓

Concatenate

[A | B]

        ↓

W_O

        ↓

Learned combination
```

## PyTorch

```python
W_O = torch.tensor([
    [1.0, 0.0, 0.5, 0.0],
    [0.0, 1.0, 0.0, 0.5],
    [0.5, 0.0, 1.0, 0.0],
    [0.0, 0.5, 0.0, 1.0]
])

attention_final = combined_heads @ W_O

print(attention_final)
print(attention_final.shape)
```

## Complete Multi-Head Attention Flow

$$
X
\rightarrow
Q,K,V
\rightarrow
\text{Split Heads}
\rightarrow
QK^T
\rightarrow
\frac{QK^T}{\sqrt{d_k}}
\rightarrow
\text{Causal Mask}
\rightarrow
\text{Softmax}
\rightarrow
\text{Attention Weights}
\rightarrow
V
\rightarrow
\text{Attention Output}
\rightarrow
\text{Combine Heads}
\rightarrow
W_O
$$

In [123]:
W_O = torch.tensor([
    [1.0, 0.0, 0.5, 0.0],
    [0.0, 1.0, 0.0, 0.5],
    [0.5, 0.0, 1.0, 0.0],
    [0.0, 0.5, 0.0, 1.0]
])

attention_final = combined_heads @ W_O

print("Final attention output:")
print(attention_final)

print("\nShape:")
print(attention_final.shape)

Final attention output:
tensor([[1.6432, 2.2822, 1.9170, 2.3735],
        [1.6432, 2.2822, 1.9170, 2.3735],
        [1.5929, 2.2689, 1.8810, 2.3651],
        [2.1840, 2.1329, 2.1633, 2.2530]])

Shape:
torch.Size([4, 4])


# 1.20.3.9 Attention Residual Connection

We have completed the Multi-Head Attention computation.

The attention output has shape:

$$
[4,4]
$$

The original input also has shape:

$$
X=[4,4]
$$

Now we add the original input back to the attention output.

## Residual Formula

$$
Y
=
X
+
\text{Attention}(\text{RMSNorm}(X))
$$

The attention path is:

$$
X
\rightarrow
\text{RMSNorm}
\rightarrow
\text{Multi-Head Attention}
$$

At the same time, the original input follows the residual path:

$$
X
\rightarrow
\text{Residual}
$$

Then both are added:

```text
              RMSNorm
                 ↓
X ────────→ Attention
│                │
│                │
└────────────────┤
                 ↓
                 +
                 ↓
                 Y
```

## Shapes

Original input:

$$
X=[4,4]
$$

Attention output:

$$
\text{Attention Output}=[4,4]
$$

Therefore:

$$
Y
=
[4,4]
+
[4,4]
=
[4,4]
$$

## Why Use a Residual Connection?

The residual connection allows the model to:

- Preserve the original token information
- Add new contextual information from attention
- Make deep networks easier to train

Conceptually:

```text
Original token representation

        +

Information gathered through attention

        ↓

Updated token representation
```

## PyTorch

```python
Y = X + attention_final

print(Y)
print(Y.shape)
```

## Transformer Flow So Far

$$
X
\rightarrow
\text{RMSNorm}
\rightarrow
\text{Multi-Head Attention}
\rightarrow
+
X
\rightarrow
Y
$$

In [124]:
Y = X + attention_final

print("Input X:")
print(X)

print("\nAttention output:")
print(attention_final)

print("\nAfter residual connection:")
print(Y)

print("\nShape:")
print(Y.shape)

Input X:
tensor([[1., 2., 3., 4.],
        [2., 4., 6., 8.],
        [1., 3., 5., 7.],
        [4., 3., 2., 1.]])

Attention output:
tensor([[1.6432, 2.2822, 1.9170, 2.3735],
        [1.6432, 2.2822, 1.9170, 2.3735],
        [1.5929, 2.2689, 1.8810, 2.3651],
        [2.1840, 2.1329, 2.1633, 2.2530]])

After residual connection:
tensor([[ 2.6432,  4.2822,  4.9170,  6.3735],
        [ 3.6432,  6.2822,  7.9170, 10.3735],
        [ 2.5929,  5.2689,  6.8810,  9.3651],
        [ 6.1840,  5.1329,  4.1633,  3.2530]])

Shape:
torch.Size([4, 4])


# 1.20.4 FFN / SwiGLU Inside the Transformer Block

After completing the Attention sub-layer and residual connection, we have:

$$
Y = X + \text{Attention}(\text{RMSNorm}(X))
$$

Now `Y` enters the second major sub-layer: the **SwiGLU Feed Forward Network**.

## High-Level Flow

$$
Y
\rightarrow
\text{RMSNorm}
\rightarrow
\text{SwiGLU}
\rightarrow
\text{Down Projection}
\rightarrow
+
Y
$$

The complete formula is:

$$
Z
=
Y
+
FFN(\text{RMSNorm}(Y))
$$

Inside the SwiGLU FFN:

$$
\text{Up}
=
Y_{norm}W_{up}
$$

$$
\text{Gate}
=
\text{SiLU}(Y_{norm}W_{gate})
$$

$$
\text{Hidden}
=
\text{Up}
\odot
\text{Gate}
$$

$$
\text{FFN Output}
=
\text{Hidden}W_{down}
$$

Then the FFN output is added back through the residual connection:

$$
Z
=
Y
+
\text{FFN Output}
$$

This completes the second major sub-layer of the Transformer block.

In [125]:
# Step 1: RMSNorm Y

Y_rms = torch.sqrt(torch.mean(Y ** 2, dim=-1, keepdim=True))
Y_norm = Y / Y_rms

print("Y:")
print(Y)

print("\nRMS per token:")
print(Y_rms)

print("\nNormalized Y:")
print(Y_norm)

print("\nShape:")
print(Y_norm.shape)

Y:
tensor([[ 2.6432,  4.2822,  4.9170,  6.3735],
        [ 3.6432,  6.2822,  7.9170, 10.3735],
        [ 2.5929,  5.2689,  6.8810,  9.3651],
        [ 6.1840,  5.1329,  4.1633,  3.2530]])

RMS per token:
tensor([[4.7466],
        [7.4670],
        [6.5104],
        [4.8089]])

Normalized Y:
tensor([[0.5569, 0.9022, 1.0359, 1.3427],
        [0.4879, 0.8413, 1.0603, 1.3892],
        [0.3983, 0.8093, 1.0569, 1.4385],
        [1.2859, 1.0674, 0.8657, 0.6764]])

Shape:
torch.Size([4, 4])


In [ ]:
(Y_norm**2).mean(dim=-1, keepdim=True) # RMS of each normalized token sum to 1

tensor([[1.0000],
        [1.0000],
        [1.0000],
        [1.0000]])

In [130]:
W_up_ffn = torch.tensor([
    [1.0, 0.0, 0.5, 0.0, 1.0, -0.5],
    [0.0, 1.0, 0.0, 0.5, -0.5, 1.0],
    [0.5, 0.0, 1.0, 0.0, 0.5, 0.5],
    [0.0, 0.5, 0.0, 1.0, 0.5, -1.0]
])

W_gate_ffn = torch.tensor([
    [0.5, 1.0, -0.5, 0.0, 0.5, 1.0],
    [1.0, -0.5, 0.5, 1.0, 0.0, -0.5],
    [0.0, 0.5, 1.0, -0.5, 1.0, 0.5],
    [0.5, 0.0, 0.5, 1.0, -0.5, 1.0]
])

up = Y_norm @ W_up_ffn
gate_raw = Y_norm @ W_gate_ffn

print("Up projection:")
print(up)

print("\nUp shape:")
print(up.shape)

print("\nGate raw:")
print(gate_raw)

print("\nGate shape:")
print(gate_raw.shape)

Up projection:
tensor([[ 1.0748,  1.5735,  1.3143,  1.7938,  1.2951, -0.2011],
        [ 1.0180,  1.5359,  1.3042,  1.8099,  1.2920, -0.2617],
        [ 0.9267,  1.5286,  1.2561,  1.8432,  1.2413, -0.2999],
        [ 1.7188,  1.4056,  1.5087,  1.2101,  1.5234,  0.1808]])

Up shape:
torch.Size([4, 6])

Gate raw:
tensor([[1.8519, 0.6237, 1.8799, 1.7269, 0.6430, 1.9665],
        [1.7799, 0.5974, 1.9316, 1.7004, 0.6096, 1.9866],
        [1.7277, 0.5221, 1.9817, 1.7193, 0.5368, 1.9606],
        [2.0486, 1.1851, 1.0947, 1.3109, 1.1705, 1.8616]])

Gate shape:
torch.Size([4, 6])


In [131]:
gate = F.silu(gate_raw)

hidden = up * gate

print("Gate after SiLU:")
print(gate)

print("\nGate shape:")
print(gate.shape)

print("\nSwiGLU hidden output:")
print(hidden)

print("\nHidden shape:")
print(hidden.shape)

Gate after SiLU:
tensor([[1.6007, 0.4061, 1.6310, 1.4662, 0.4214, 1.7250],
        [1.5230, 0.3853, 1.6871, 1.4379, 0.3949, 1.7470],
        [1.4670, 0.3277, 1.7416, 1.4581, 0.3388, 1.7186],
        [1.8146, 0.9077, 0.8202, 1.0326, 0.8934, 1.6112]])

Gate shape:
torch.Size([4, 6])

SwiGLU hidden output:
tensor([[ 1.7205,  0.6390,  2.1437,  2.6301,  0.5458, -0.3468],
        [ 1.5505,  0.5919,  2.2003,  2.6024,  0.5102, -0.4572],
        [ 1.3596,  0.5009,  2.1876,  2.6875,  0.4205, -0.5154],
        [ 3.1190,  1.2758,  1.2374,  1.2496,  1.3609,  0.2913]])

Hidden shape:
torch.Size([4, 6])


In [132]:
# Next: Down Projection
W_down_ffn = torch.tensor([
    [1.0, 0.0, 0.5, 0.0],
    [0.0, 1.0, 0.0, 0.5],
    [0.5, 0.0, 1.0, 0.0],
    [0.0, 0.5, 0.0, 1.0],
    [0.5, 0.5, 0.5, 0.0],
    [0.0, 0.0, 0.5, 0.5]
])

ffn_output = hidden @ W_down_ffn

print("FFN output:")
print(ffn_output)

print("\nShape:")
print(ffn_output.shape)

FFN output:
tensor([[3.0652, 2.2269, 3.1034, 2.7762],
        [2.9058, 2.1482, 3.0021, 2.6697],
        [2.6636, 2.0549, 2.8200, 2.6802],
        [4.4182, 2.5810, 3.6230, 2.0331]])

Shape:
torch.Size([4, 4])


In [ ]:
import torch
a = torch.arange(32, dtype=torch.float).reshape(4,8)
b = a.view(4,4,2)

# c = torch.arange(0,1, 0.065).reshape(4,4)
b[:,:,0]

tensor([[[ 0.,  1.],
         [ 2.,  3.],
         [ 4.,  5.],
         [ 6.,  7.]],

        [[ 8.,  9.],
         [10., 11.],
         [12., 13.],
         [14., 15.]],

        [[16., 17.],
         [18., 19.],
         [20., 21.],
         [22., 23.]],

        [[24., 25.],
         [26., 27.],
         [28., 29.],
         [30., 31.]]])

In [37]:
x = torch.randn(1, 2, 4, 8)
y = x.view(1,2,4,4,2)
y

tensor([[[[[ 0.6615,  0.3168],
           [-2.1125, -0.5156],
           [-1.3691,  0.6228],
           [-1.2559,  0.9998]],

          [[ 1.3146,  0.5892],
           [ 0.9000, -0.0093],
           [-0.4237, -1.3189],
           [-0.2885,  0.9349]],

          [[-0.8859,  1.6191],
           [-0.3850, -1.4102],
           [-0.0347, -1.0487],
           [-0.5202,  1.5458]],

          [[-0.5512,  0.1665],
           [-0.0876,  1.9659],
           [-1.5573, -0.5148],
           [ 0.8379, -0.0263]]],


         [[[ 0.7529,  1.1298],
           [ 0.2546,  0.1917],
           [-0.8062, -1.3507],
           [-1.5348, -0.8819]],

          [[ 0.8982, -0.7056],
           [ 0.4375,  0.5655],
           [-1.2787, -0.5064],
           [-2.4851,  0.3139]],

          [[ 1.7053, -0.2119],
           [ 0.9404, -0.2146],
           [-1.3776, -1.1393],
           [-0.8988, -0.3828]],

          [[ 0.6747,  1.0052],
           [ 1.0635, -0.5233],
           [ 1.6026, -0.6940],
           [-1.8464,  0

In [41]:
y1 = y[:,:,:,:,0]
y2 = y[:,:,:,:,1]
y1,y2,y1.shape, y2.shape

(tensor([[[[ 0.6615, -2.1125, -1.3691, -1.2559],
           [ 1.3146,  0.9000, -0.4237, -0.2885],
           [-0.8859, -0.3850, -0.0347, -0.5202],
           [-0.5512, -0.0876, -1.5573,  0.8379]],
 
          [[ 0.7529,  0.2546, -0.8062, -1.5348],
           [ 0.8982,  0.4375, -1.2787, -2.4851],
           [ 1.7053,  0.9404, -1.3776, -0.8988],
           [ 0.6747,  1.0635,  1.6026, -1.8464]]]]),
 tensor([[[[ 0.3168, -0.5156,  0.6228,  0.9998],
           [ 0.5892, -0.0093, -1.3189,  0.9349],
           [ 1.6191, -1.4102, -1.0487,  1.5458],
           [ 0.1665,  1.9659, -0.5148, -0.0263]],
 
          [[ 1.1298,  0.1917, -1.3507, -0.8819],
           [-0.7056,  0.5655, -0.5064,  0.3139],
           [-0.2119, -0.2146, -1.1393, -0.3828],
           [ 1.0052, -0.5233, -0.6940,  0.9983]]]]),
 torch.Size([1, 2, 4, 4]),
 torch.Size([1, 2, 4, 4]))

In [49]:
z = torch.stack([y1, y2], dim= -1)
z, z.shape

(tensor([[[[[ 0.6615,  0.3168],
            [-2.1125, -0.5156],
            [-1.3691,  0.6228],
            [-1.2559,  0.9998]],
 
           [[ 1.3146,  0.5892],
            [ 0.9000, -0.0093],
            [-0.4237, -1.3189],
            [-0.2885,  0.9349]],
 
           [[-0.8859,  1.6191],
            [-0.3850, -1.4102],
            [-0.0347, -1.0487],
            [-0.5202,  1.5458]],
 
           [[-0.5512,  0.1665],
            [-0.0876,  1.9659],
            [-1.5573, -0.5148],
            [ 0.8379, -0.0263]]],
 
 
          [[[ 0.7529,  1.1298],
            [ 0.2546,  0.1917],
            [-0.8062, -1.3507],
            [-1.5348, -0.8819]],
 
           [[ 0.8982, -0.7056],
            [ 0.4375,  0.5655],
            [-1.2787, -0.5064],
            [-2.4851,  0.3139]],
 
           [[ 1.7053, -0.2119],
            [ 0.9404, -0.2146],
            [-1.3776, -1.1393],
            [-0.8988, -0.3828]],
 
           [[ 0.6747,  1.0052],
            [ 1.0635, -0.5233],
            [ 1.

In [50]:
z.view(1,2,4,8)

tensor([[[[ 0.6615,  0.3168, -2.1125, -0.5156, -1.3691,  0.6228, -1.2559,
            0.9998],
          [ 1.3146,  0.5892,  0.9000, -0.0093, -0.4237, -1.3189, -0.2885,
            0.9349],
          [-0.8859,  1.6191, -0.3850, -1.4102, -0.0347, -1.0487, -0.5202,
            1.5458],
          [-0.5512,  0.1665, -0.0876,  1.9659, -1.5573, -0.5148,  0.8379,
           -0.0263]],

         [[ 0.7529,  1.1298,  0.2546,  0.1917, -0.8062, -1.3507, -1.5348,
           -0.8819],
          [ 0.8982, -0.7056,  0.4375,  0.5655, -1.2787, -0.5064, -2.4851,
            0.3139],
          [ 1.7053, -0.2119,  0.9404, -0.2146, -1.3776, -1.1393, -0.8988,
           -0.3828],
          [ 0.6747,  1.0052,  1.0635, -0.5233,  1.6026, -0.6940, -1.8464,
            0.9983]]]])

In [51]:
x

tensor([[[[ 0.6615,  0.3168, -2.1125, -0.5156, -1.3691,  0.6228, -1.2559,
            0.9998],
          [ 1.3146,  0.5892,  0.9000, -0.0093, -0.4237, -1.3189, -0.2885,
            0.9349],
          [-0.8859,  1.6191, -0.3850, -1.4102, -0.0347, -1.0487, -0.5202,
            1.5458],
          [-0.5512,  0.1665, -0.0876,  1.9659, -1.5573, -0.5148,  0.8379,
           -0.0263]],

         [[ 0.7529,  1.1298,  0.2546,  0.1917, -0.8062, -1.3507, -1.5348,
           -0.8819],
          [ 0.8982, -0.7056,  0.4375,  0.5655, -1.2787, -0.5064, -2.4851,
            0.3139],
          [ 1.7053, -0.2119,  0.9404, -0.2146, -1.3776, -1.1393, -0.8988,
           -0.3828],
          [ 0.6747,  1.0052,  1.0635, -0.5233,  1.6026, -0.6940, -1.8464,
            0.9983]]]])

# Language Model Training: Target Shifting

A language model learns to predict the next token.

Given an original token sequence:

[1, 2, 3, 4, 5]

We shift it to create:

Input:  [1, 2, 3, 4]
Target: [2, 3, 4, 5]

The model receives the input tokens and produces logits for each position.
These logits are then compared with the corresponding target tokens
using CrossEntropyLoss.

In [53]:
import torch

tokens = torch.tensor([
    [1, 2, 3, 4, 5],
    [6, 7, 8, 9, 10]
])

inputs = tokens[:, :-1]
targets = tokens[:, 1:]

print("Original tokens:")
print(tokens)

print("\nInputs:")
print(inputs)

print("\nTargets:")
print(targets)

print("\nInput shape:", inputs.shape)
print("Target shape:", targets.shape)

Original tokens:
tensor([[ 1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10]])

Inputs:
tensor([[1, 2, 3, 4],
        [6, 7, 8, 9]])

Targets:
tensor([[ 2,  3,  4,  5],
        [ 7,  8,  9, 10]])

Input shape: torch.Size([2, 4])
Target shape: torch.Size([2, 4])


In [54]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [59]:
from smollm2.model import SmolLM2

In [60]:
from smollm2.model import SmolLM2

model = SmolLM2(
    vocab_size=100,
    hidden_size=8,
    num_heads=2,
    intermediate_size=32,
    num_layers=2
)

logits = model(inputs)

print("Input shape:", inputs.shape)
print("Target shape:", targets.shape)
print("Logits shape:", logits.shape)

Input shape: torch.Size([2, 4])
Target shape: torch.Size([2, 4])
Logits shape: torch.Size([2, 4, 100])


In [66]:
logits_flat = logits.reshape(-1, 100)
targets_flat = targets.reshape(-1)

print("Original logits:", logits.shape)
print("Flattened logits:", logits_flat.shape)

print("\nOriginal targets:", targets.shape)
print("Flattened targets:", targets_flat.shape)

Original logits: torch.Size([2, 4, 100])
Flattened logits: torch.Size([8, 100])

Original targets: torch.Size([2, 4])
Flattened targets: torch.Size([8])


In [68]:
loss_fn = torch.nn.CrossEntropyLoss()
loss = loss_fn(logits_flat, targets_flat)
loss

tensor(4.7617, grad_fn=<NllLossBackward0>)

In [69]:
import torch

logits = torch.tensor([[2.0, 1.0, 0.1]])
target = torch.tensor([1])

print("Logits:", logits)
print("Target:", target)

Logits: tensor([[2.0000, 1.0000, 0.1000]])
Target: tensor([1])


In [70]:
probs = torch.softmax(logits, dim=-1)

print("Probabilities:", probs)
print("Sum:", probs.sum())

Probabilities: tensor([[0.6590, 0.2424, 0.0986]])
Sum: tensor(1.0000)


In [71]:
target_prob = probs[0, target.item()]

print("Target:", target.item())
print("Probability assigned to correct target:", target_prob.item())

Target: 1
Probability assigned to correct target: 0.24243298172950745


In [74]:
manual_loss = -torch.log(target_prob)
print("Manual loss:", manual_loss.item())

Manual loss: 1.4170299768447876


In [75]:
loss_fn = torch.nn.CrossEntropyLoss()

pytorch_loss = loss_fn(logits, target)

print("PyTorch loss:", pytorch_loss.item())

PyTorch loss: 1.4170299768447876


### example with 2 predictions adn 3 tokens each

In [76]:
logits = torch.tensor([
    [2.0, 1.0, 0.1],
    [0.5, 2.5, 1.0]
])

targets = torch.tensor([1, 2])

In [77]:
loss_fn = torch.nn.CrossEntropyLoss()
loss = loss_fn(logits, targets)
loss

tensor(1.6117)

In [ ]:
probs = torch.softmax(logits, dim= 1)
probs

tensor([[0.6590, 0.2424, 0.0986],
        [0.0996, 0.7361, 0.1643]])

In [89]:
target_prob = probs[range(len(targets)), targets]
target_prob


tensor([0.2424, 0.1643])

In [91]:
loss = torch.log(target_prob)
loss = -loss.sum() / len(targets)
loss

tensor(1.6117)

In [88]:
range(len(targets))

range(0, 2)

In [97]:
import torch
input = 3

weight = torch.tensor(2.0, requires_grad=True)

loss = weight ** 2

print("Weight:", weight)
print("Loss:", loss)

Weight: tensor(2., requires_grad=True)
Loss: tensor(4., grad_fn=<PowBackward0>)


In [98]:
loss.backward()

print(weight.grad)

tensor(4.)


In [99]:
w_n = weight - 0.1 * weight.grad
w_n

tensor(1.6000, grad_fn=<SubBackward0>)

In [100]:
optimizer = torch.optim.SGD([weight], lr=0.1)
optimizer.step()

print(weight)

tensor(1.6000, requires_grad=True)


In [136]:
model = SmolLM2(
    vocab_size=100,
    hidden_size=8,
    num_heads=2,
    intermediate_size=32,
    num_layers=2
)

In [137]:
import torch

input_ids = torch.tensor([
    [10, 20, 30, 40],
    [50, 60, 70, 80]
])

target = torch.tensor([
    [20, 30, 40, 50],
    [60, 70, 80, 90]
])

In [138]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)
for _ in range(20):

    # Forward pass
    prediction = model(input_ids)

    # Calculate loss
    loss = loss_fn(prediction.transpose(1,2), target)

    # Clear old gradients
    optimizer.zero_grad()

    # Calculate new gradients
    loss.backward()

    # Update weights
    optimizer.step()

    if _ % 10 == 0:
        print(f"Loop: {_}, Loss: {loss.item():.4f}")

Loop: 0, Loss: 4.8407
Loop: 10, Loss: 2.9853


In [139]:
prediction.argmax(dim=-1)

tensor([[20, 30, 40, 50],
        [60, 70, 80, 90]])

In [140]:
target

tensor([[20, 30, 40, 50],
        [60, 70, 80, 90]])

### Tokenizer

In [142]:
a = [1,2,3,4,3,2,1]
list(set(a))

[1, 2, 3, 4]

In [144]:
from data.sample import *

ModuleNotFoundError: No module named 'data.sample'

In [143]:
import re
# Read training text from data/sample.txt
with open('data/sample.txt', 'r') as file:
    text = file.read()

text = text.lower() # conver lowercase
tokens = re.findall(r"\w+|[^\w\s]", text) # split the text

vocab = sorted(set(tokens)) # create unique vocab

# create token to id mapping
token_to_id = {token:idx for idx, token in enumerate(vocab)}

# create id to token mapping
id_to_token = {idx:token for idx, token in enumerate(vocab)}

# convert the token into ids
token_ids = [token_to_id[token] for token in tokens]

# convert ids back to tokens
decoded_tokens = [id_to_token[idx] for idx in token_ids]

full_text = ' '.join(decoded_tokens).replace(' .', '.')
# full_text = full_text.replace('. ', '.\n')
print(full_text)


FileNotFoundError: [Errno 2] No such file or directory: 'data/sample.txt'

In [153]:
numbers = [2,4,6,8]

for num in numbers:
    if num % 2 == 0:
        numbers.remove(num)
    print(numbers)

print(numbers)


[4, 6, 8]
[4, 8]
[4, 8]
